[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/calculus/02_limits_and_continuity/exercises.ipynb)

# Module 02 — Exercises: Limits and Continuity

Forty-five solved problems in four tiers. Every problem carries a statement, a one-line
intuition, a stepwise solution, a boxed answer, a key takeaway, and — wherever the answer is
numeric or algorithmic — a code cell that recomputes it and prints the check.

Definition, theorem, proof and example numbers refer to
[first_principles.ipynb](first_principles.ipynb). Symbols follow
[the notation register](../../docs/notation.md): the limit quantifiers are $\varepsilon$ and
$\delta$, absolute values are written $\lvert \cdot \rvert$, and the floating-point constants are
$\varepsilon_{\mathrm{mach}} = 2^{-52}$ with unit roundoff $u = \tfrac{1}{2}\varepsilon_{\mathrm{mach}} = 2^{-53}$.

The preamble below is shared by every code cell in this notebook.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (7.0, 4.0),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})
rng = np.random.default_rng(0)
np.set_printoptions(precision=4, suppress=True)

EPS = np.finfo(float).eps          # machine epsilon, 2^-52
UNIT_ROUNDOFF = 0.5 * EPS          # u = 2^-53
print(f"machine epsilon eps_mach = {EPS:.6e}")
print(f"unit roundoff   u        = {UNIT_ROUNDOFF:.6e}")

machine epsilon eps_mach = 2.220446e-16
unit roundoff   u        = 1.110223e-16


## L0 — Concept Checks

### Problem L0.1 — Negating the limit definition

**Statement.** Write the formal negation of $\lim_{x \to a} f(x) = L$ as given in Definition 3.1,
and say in one sentence what an adversary must exhibit to refute the limit.

**Intuition.** Negating a $\forall \exists \forall$ statement flips every quantifier and turns the
implication into a conjunction.

**Solution.**

*Step 1 — the statement.* Definition 3.1 reads

$$
\forall \varepsilon \gt 0 \ \ \exists \delta \gt 0 \ \ \forall x \in D : \ 0 \lt \lvert x - a \rvert \lt \delta \implies \lvert f(x) - L \rvert \lt \varepsilon .
$$

*Step 2 — flip the quantifiers.* $\neg \forall \to \exists$, $\neg \exists \to \forall$, and
$\neg (P \implies Q) \equiv P \wedge \neg Q$.

$$
\boxed{\exists \varepsilon \gt 0 \ \ \forall \delta \gt 0 \ \ \exists x \in D : \ 0 \lt \lvert x - a \rvert \lt \delta \ \text{ and } \ \lvert f(x) - L \rvert \ge \varepsilon}
$$

**Key takeaway.** One bad $\varepsilon$ destroys a limit; producing it is exactly the
$(\Leftarrow)$ half of Proof 5.4, which converts that single $\varepsilon_0$ into a sequence.

### Problem L0.2 — The sign function has no two-sided limit

**Statement.** For $f(x) = \lvert x \rvert / x$ on $x \neq 0$, evaluate the two one-sided limits
at $0$ and decide whether $\lim_{x \to 0} f(x)$ exists.

**Intuition.** The function is $+1$ to the right of the origin and $-1$ to the left, so the two
half-neighbourhoods disagree.

**Solution.**

*Step 1.* For $x \gt 0$, $\lvert x \rvert = x$ and $f(x) = 1$, so $\lim_{x \to 0^{+}} f(x) = 1$.

*Step 2.* For $x \lt 0$, $\lvert x \rvert = -x$ and $f(x) = -1$, so $\lim_{x \to 0^{-}} f(x) = -1$.

*Step 3.* Definition 3.2 makes the two-sided limit equal to the common value of the one-sided
limits; here they differ.

$$
\boxed{\lim_{x \to 0^{+}} f(x) = 1, \quad \lim_{x \to 0^{-}} f(x) = -1, \quad \lim_{x \to 0} f(x) \ \text{does not exist}}
$$

**Key takeaway.** By Definition 3.6 this is a jump discontinuity, of size
$\lvert L_{+} - L_{-} \rvert = 2$.

In [2]:
def sgn(x):
    return np.abs(x) / x


for h in (1e-1, 1e-3, 1e-6, 1e-12):
    print(f"  h = {h:.0e}   f(+h) = {sgn(h):+.1f}   f(-h) = {sgn(-h):+.1f}")
print("jump size |L+ - L-| =", sgn(1e-12) - sgn(-1e-12))
assert sgn(1e-12) == 1.0 and sgn(-1e-12) == -1.0

  h = 1e-01   f(+h) = +1.0   f(-h) = -1.0
  h = 1e-03   f(+h) = +1.0   f(-h) = -1.0
  h = 1e-06   f(+h) = +1.0   f(-h) = -1.0
  h = 1e-12   f(+h) = +1.0   f(-h) = -1.0
jump size |L+ - L-| = 2.0


### Problem L0.3 — The value at the point is irrelevant

**Statement.** Let $f(x) = x^{2}$ for $x \neq 2$ and $f(2) = 7$. Find $\lim_{x \to 2} f(x)$,
compare it with $f(2)$, and classify the point.

**Intuition.** Definition 3.1 punches out $x = a$, so $f(2)$ never enters the computation.

**Solution.**

*Step 1.* On the punctured neighbourhood $f$ agrees with $x^{2}$, so
$\lim_{x \to 2} f(x) = 4$.

*Step 2.* $f(2) = 7 \neq 4$, so Definition 3.4 fails at $x = 2$.

*Step 3.* The limit exists and is finite, so Definition 3.6 calls this removable: redefining
$f(2) = 4$ repairs continuity.

$$
\boxed{\lim_{x \to 2} f(x) = 4 \neq 7 = f(2); \ \text{removable discontinuity}}
$$

**Key takeaway.** A removable discontinuity is a defect of one point, not of the function's
behaviour near it.

In [3]:
def f_L03(x):
    return np.where(x == 2.0, 7.0, x ** 2)


probe = 2.0 + np.array([-1e-6, -1e-9, 1e-9, 1e-6])
print("f near 2 :", f_L03(probe))
print("f(2)     :", float(f_L03(np.array([2.0]))[0]))
assert np.allclose(f_L03(probe), probe ** 2)
assert float(f_L03(np.array([2.0]))[0]) == 7.0

f near 2 : [4. 4. 4. 4.]
f(2)     : 7.0


### Problem L0.4 — Two discontinuities at the same point

**Statement.** Classify the discontinuity at $x = 1$ of

$$
f(x) = \frac{x^{2} - 1}{x - 1}, \qquad g(x) = \frac{1}{(x-1)^{2}} .
$$

**Intuition.** One factor cancels; the other does not.

**Solution.**

*Step 1.* For $x \neq 1$, $f(x) = \dfrac{(x-1)(x+1)}{x-1} = x + 1$, so
$\lim_{x \to 1} f(x) = 2$ while $f(1)$ is undefined: removable.

*Step 2.* As $x \to 1$, $(x-1)^{2} \to 0^{+}$, so $g(x) \to +\infty$ in the sense of
Definition 3.3.

*Step 3.* By Definition 3.6 that is an essential discontinuity of the *infinite* sub-type.

$$
\boxed{f: \ \text{removable, limit } 2; \qquad g: \ \text{essential (infinite)}}
$$

**Key takeaway.** "Essential" is the umbrella for everything that is neither removable nor a jump;
infinite and oscillatory are its two sub-cases.

In [4]:
h = np.array([1e-2, 1e-4, 1e-6, 1e-8])
print("f(1+h) = (x^2-1)/(x-1) :", ((1 + h) ** 2 - 1) / h)
print("g(1+h) = 1/(x-1)^2     :", 1.0 / h ** 2)
assert np.allclose(((1 + h) ** 2 - 1) / h, 2 + h)
assert np.all(1.0 / h ** 2 > 1e3)

f(1+h) = (x^2-1)/(x-1) : [2.01   2.0001 2.     2.    ]
g(1+h) = 1/(x-1)^2     : [1.e+04 1.e+08 1.e+12 1.e+16]


### Problem L0.5 — A squeeze that pins the value as well as the limit

**Statement.** Suppose $-x^{2} \le f(x) \le x^{2}$ for all $x \in [-1, 1]$. Determine
$\lim_{x \to 0} f(x)$ and $f(0)$.

**Intuition.** The two envelopes meet at the origin, so nothing is left for $f$ to do there.

**Solution.**

*Step 1.* $\lim_{x \to 0}(-x^{2}) = \lim_{x \to 0} x^{2} = 0$, so Theorem 4.3 gives
$\lim_{x \to 0} f(x) = 0$.

*Step 2.* At $x = 0$ the hypothesis reads $0 \le f(0) \le 0$, which forces $f(0) = 0$ — this step
uses the inequality *at* the point, which the limit statement does not.

$$
\boxed{\lim_{x \to 0} f(x) = 0 \ \text{ and } \ f(0) = 0, \ \text{so } f \text{ is continuous at } 0}
$$

**Key takeaway.** The squeeze delivers existence, not just a value; no candidate $L$ has to be
guessed first.

In [5]:
xs = np.linspace(-1.0, 1.0, 20001)
xs = xs[xs != 0.0]
f_member = xs ** 2 * np.sin(37.0 / xs)          # one function inside the envelope
print("max |f| on |x| < 0.01 :", np.abs(f_member[np.abs(xs) < 0.01]).max())
print("envelope value x^2    :", 0.01 ** 2)
assert np.all(np.abs(f_member) <= xs ** 2 + 1e-18)
assert np.abs(f_member[np.abs(xs) < 0.01]).max() <= 1e-4

max |f| on |x| < 0.01 : 8.82836652288846e-05
envelope value x^2    : 0.0001


### Problem L0.6 — The Intermediate Value Theorem needs a connected domain

**Statement.** $f(x) = 1/x$ on $D = [-1, 0) \cup (0, 1]$ has $f(-1) = -1$ and $f(1) = 1$, yet no
root. Which hypothesis of Theorem 4.5 fails?

**Intuition.** $D$ is not an interval; the sign change happens across a hole.

**Solution.**

*Step 1.* $d = 0$ lies strictly between $f(-1) = -1$ and $f(1) = 1$.

*Step 2.* $1/c = 0$ has no solution, so the conclusion of Theorem 4.5 is false here.

*Step 3.* Theorem 4.5 requires continuity on the **whole** closed interval $[-1, 1]$. The point
$x = 0$ is not in $D$ at all, so $f$ is not a continuous function on $[-1,1]$ and the theorem does
not apply.

$$
\boxed{\text{The domain is not an interval: } f \ \text{is not continuous on } [-1,1], \ \text{so Theorem 4.5 does not apply}}
$$

**Key takeaway.** IVT is a statement about connectedness. Continuity on each piece of a
disconnected domain buys nothing.

In [6]:
grid = np.concatenate([np.linspace(-1.0, -1e-9, 50000), np.linspace(1e-9, 1.0, 50000)])
vals = 1.0 / grid
print("f(-1), f(1)                    :", 1 / -1.0, 1 / 1.0)
print("smallest |f| anywhere on D     :", np.abs(vals).min())
assert np.abs(vals).min() >= 1.0

f(-1), f(1)                    : -1.0 1.0
smallest |f| anywhere on D     : 1.0


### Problem L0.7 — The Extreme Value Theorem needs a compact domain

**Statement.** Give a function continuous on $(0,1)$ with no maximum, and name the hypothesis of
Theorem 4.7 that fails.

**Intuition.** An open interval lets the function escape through the missing endpoint.

**Solution.**

*Step 1.* Take $f(x) = 1/x$ on $(0,1)$, continuous at every point of that set.

*Step 2.* As $x \to 0^{+}$, $f(x) \to +\infty$ by Definition 3.3, so $f$ is unbounded above and
has no maximum.

*Step 3.* Theorem 4.7 asks for a closed bounded interval $[a,b]$; $(0,1)$ is not closed.

$$
\boxed{f(x) = 1/x \ \text{on } (0,1) \ \text{is continuous and unbounded; the domain is not closed}}
$$

**Key takeaway.** Even the milder $f(x) = x$ on $[0,1)$ fails: it is bounded and still attains no
maximum, so closedness costs attainment before it costs boundedness.

In [7]:
x_open = np.linspace(1e-6, 1 - 1e-9, 200000)
print("sup 1/x on (0,1) sampled  :", (1.0 / x_open).max(), " (grows without bound)")
x_half = np.linspace(0.0, 1 - 1e-9, 200000)
print("sup x on [0,1) sampled    :", x_half.max(), " < 1, never attained")
assert x_half.max() < 1.0
assert (1.0 / x_open).max() > 1e5

sup 1/x on (0,1) sampled  : 1000000.0  (grows without bound)
sup x on [0,1) sampled    : 0.999999999  < 1, never attained


### Problem L0.8 — A growth hierarchy at infinity

**Statement.** Order $\sqrt{n}$, $n \ln n$, $n^{2}$, $e^{n}$, $n!$ by growth as $n \to \infty$
through the integers, using Definition 3.8.

**Intuition.** Roots lose to logarithm-boosted linear growth, which loses to powers, which lose to
exponentials, which lose to factorials.

**Solution.**

*Step 1.* $\dfrac{\sqrt{n}}{n \ln n} = \dfrac{1}{\sqrt{n}\,\ln n} \to 0$.

*Step 2.* $\dfrac{n \ln n}{n^{2}} = \dfrac{\ln n}{n} \to 0$.

*Step 3.* $\dfrac{n^{2}}{e^{n}} \to 0$.

*Step 4.* $\dfrac{e^{n}}{n!} \to 0$, since for $n \ge 6$ the ratio of consecutive terms is
$e/(n+1) \lt \tfrac{1}{2}$.

*Step 5 — the domain matters.* $n!$ is defined only on $\mathbb{N}$, so the chain is a statement
about integer $n$. For real $x$ replace it by $\Gamma(x+1)$, whose Stirling asymptotic
$\Gamma(x+1) \sim \sqrt{2\pi x}\,(x/e)^{x}$ gives the same ordering.

$$
\boxed{\sqrt{n} = o(n \ln n) = o(n^{2}) = o(e^{n}) = o(n!) \qquad (n \to \infty, \ n \in \mathbb{N})}
$$

**Key takeaway.** Each "$\ll$" is a limit of a ratio, and each is directional: $x = O(x^{2})$ holds
at infinity and fails at the origin.

In [8]:
from math import lgamma

ns = np.array([5, 10, 20, 50, 100], dtype=float)
log_terms = {
    "sqrt(n)": 0.5 * np.log(ns),
    "n ln n": np.log(ns) + np.log(np.log(ns)),
    "n^2": 2 * np.log(ns),
    "e^n": ns,
    "n! (lgamma)": np.array([lgamma(n + 1) for n in ns]),
}
print("logarithms of the five growth functions")
for name, v in log_terms.items():
    print(f"  {name:<12}", np.array2string(v, precision=3))
order = list(log_terms)
for a, b in zip(order[:-1], order[1:]):
    gap = log_terms[b] - log_terms[a]
    print(f"  log({b}) - log({a}) at n = 100 : {gap[-1]:.3f}   (increasing: {bool(np.all(np.diff(gap) > 0))})")
    assert gap[-1] > 0 and np.all(np.diff(gap) > 0)

logarithms of the five growth functions
  sqrt(n)      [0.805 1.151 1.498 1.956 2.303]
  n ln n       [2.085 3.137 4.093 5.276 6.132]
  n^2          [3.219 4.605 5.991 7.824 9.21 ]
  e^n          [  5.  10.  20.  50. 100.]
  n! (lgamma)  [  4.787  15.104  42.336 148.478 363.739]
  log(n ln n) - log(sqrt(n)) at n = 100 : 3.830   (increasing: True)
  log(n^2) - log(n ln n) at n = 100 : 3.078   (increasing: True)
  log(e^n) - log(n^2) at n = 100 : 90.790   (increasing: True)
  log(n! (lgamma)) - log(e^n) at n = 100 : 263.739   (increasing: True)


### Problem L0.9 — One-sided limits of the fractional part

**Statement.** Evaluate $\lim_{x \to 2^{+}} (x - \lfloor x \rfloor)$ and
$\lim_{x \to 2^{-}} (x - \lfloor x \rfloor)$.

**Intuition.** The floor jumps by one as $x$ crosses an integer, and the fractional part drops
back to zero.

**Solution.**

*Step 1.* For $x \in (2,3)$, $\lfloor x \rfloor = 2$, so the expression is $x - 2 \to 0$.

*Step 2.* For $x \in (1,2)$, $\lfloor x \rfloor = 1$, so the expression is $x - 1 \to 1$.

$$
\boxed{\lim_{x \to 2^{+}} (x - \lfloor x \rfloor) = 0, \qquad \lim_{x \to 2^{-}} (x - \lfloor x \rfloor) = 1}
$$

**Key takeaway.** A jump of size $1$ at every integer; by Proposition 4.12 a non-decreasing
function could only have jumps, and $x - \lfloor x \rfloor$ shows that a function which is not
monotone can still have exactly the same discontinuity type.

In [9]:
def frac(x):
    return x - np.floor(x)


for h in (1e-2, 1e-6, 1e-12):
    print(f"  h = {h:.0e}   frac(2+h) = {frac(2 + h):.12f}   frac(2-h) = {frac(2 - h):.12f}")
assert abs(frac(2 + 1e-12)) < 1e-11
assert abs(frac(2 - 1e-12) - 1.0) < 1e-11

  h = 1e-02   frac(2+h) = 0.010000000000   frac(2-h) = 0.990000000000
  h = 1e-06   frac(2+h) = 0.000001000000   frac(2-h) = 0.999999000000
  h = 1e-12   frac(2+h) = 0.000000000001   frac(2-h) = 0.999999999999


### Problem L0.10 — Upper and lower limits of $\sin(1/x)$ at the origin

**Statement.** Using Definition 3.5, compute $\limsup_{x \to 0} \sin(1/x)$ and
$\liminf_{x \to 0} \sin(1/x)$, and name the class Definition 3.6 assigns to $x = 0$.

**Intuition.** However small the punctured neighbourhood, $1/x$ still runs over an unbounded set
of angles, so the sine attains both extremes inside it.

**Solution.**

*Step 1.* Fix $r \gt 0$. For every integer $n$ with $2\pi n + \tfrac{\pi}{2} \gt 1/r$ the point
$x = (2\pi n + \tfrac{\pi}{2})^{-1}$ satisfies $0 \lt x \lt r$ and $\sin(1/x) = 1$. Hence the
supremum over the punctured neighbourhood is $1$ for every $r$.

*Step 2.* The same argument with $2\pi n - \tfrac{\pi}{2}$ gives infimum $-1$ for every $r$.

*Step 3.* Taking $r \to 0^{+}$ in Definition 3.5 leaves the constants unchanged.

*Step 4.* The two differ, so no limit exists; the function is bounded, so nothing is infinite.

$$
\boxed{\limsup_{x \to 0} \sin\frac{1}{x} = 1, \quad \liminf_{x \to 0} \sin\frac{1}{x} = -1, \quad \text{oscillatory essential discontinuity}}
$$

**Key takeaway.** $\limsup = \liminf$ is exactly the condition for a limit to exist, and the gap
between them measures how badly it fails.

In [10]:
for r in (1e-1, 1e-3, 1e-6):
    xs = np.linspace(-r, r, 400001)
    xs = xs[xs != 0.0]
    v = np.sin(1.0 / xs)
    print(f"  r = {r:.0e}   sup = {v.max():.10f}   inf = {v.min():.10f}")
    assert v.max() > 0.999999 and v.min() < -0.999999
print("both stay pinned at +/-1 as r shrinks: limsup = 1, liminf = -1")

  r = 1e-01   sup = 1.0000000000   inf = -1.0000000000
  r = 1e-03   sup = 1.0000000000   inf = -1.0000000000
  r = 1e-06   sup = 0.9999999998   inf = -0.9999999998
both stay pinned at +/-1 as r shrinks: limsup = 1, liminf = -1


## L1 — Foundations

### Problem L1.1 — An $\varepsilon$-$\delta$ proof for a linear function

**Statement.** Prove from Definition 3.1 that $\lim_{x \to 2} (5x - 3) = 7$, exhibiting
$\delta(\varepsilon)$ explicitly.

**Intuition.** A linear function magnifies the input error by exactly its slope, so dividing by
the slope undoes it.

**Solution.**

*Step 1 — factor the error.*

$$
\lvert (5x - 3) - 7 \rvert = \lvert 5x - 10 \rvert = 5 \lvert x - 2 \rvert .
$$

*Step 2 — solve for $\delta$.* Demanding $5\lvert x - 2 \rvert \lt \varepsilon$ gives
$\lvert x - 2 \rvert \lt \varepsilon/5$.

*Step 3 — verify.* If $0 \lt \lvert x - 2 \rvert \lt \varepsilon/5$ then
$\lvert (5x-3) - 7 \rvert = 5 \lvert x - 2 \rvert \lt \varepsilon$, and $\delta$ mentions only
$\varepsilon$.

$$
\boxed{\delta(\varepsilon) = \frac{\varepsilon}{5}}
$$

**Key takeaway.** For $f(x) = mx + b$ with $m \neq 0$ the certificate is always
$\delta = \varepsilon / \lvert m \rvert$, and it is sharp: no larger $\delta$ works.

In [11]:
eps = 1e-3
delta = eps / 5
xs = np.linspace(2 - delta, 2 + delta, 200001)
worst = np.abs((5 * xs - 3) - 7).max()
print(f"delta = eps/5 = {delta:.8e}")
print(f"sup |(5x-3) - 7| on the strip = {worst:.8e}   (must be < eps = {eps:.0e})")
print(f"ratio worst/eps = {worst/eps:.6f}   (1 means the certificate is sharp)")
assert worst <= eps

delta = eps/5 = 2.00000000e-04
sup |(5x-3) - 7| on the strip = 1.00000000e-03   (must be < eps = 1e-03)
ratio worst/eps = 1.000000   (1 means the certificate is sharp)


### Problem L1.2 — An $\varepsilon$-$\delta$ proof for $x^{2}$

**Statement.** Prove from Definition 3.1 that $\lim_{x \to 3} x^{2} = 9$.

**Intuition.** The error factors as $\lvert x-3 \rvert \lvert x+3 \rvert$; capping $\delta$ first
turns the second factor into a constant.

**Solution.**

*Step 1 — factor.* $\lvert x^{2} - 9 \rvert = \lvert x - 3 \rvert \, \lvert x + 3 \rvert$.

*Step 2 — cap.* Agree that $\delta \le 1$. Then $2 \lt x \lt 4$, so $\lvert x + 3 \rvert \lt 7$.

*Step 3 — solve.* Under the cap, $\lvert x^{2} - 9 \rvert \lt 7 \lvert x - 3 \rvert$, so
$\lvert x - 3 \rvert \lt \varepsilon/7$ suffices.

$$
\boxed{\delta(\varepsilon) = \min\left(1, \frac{\varepsilon}{7}\right)}
$$

**Key takeaway.** This is Example 6.1 in general form; the sharp $\delta$ is
$\min(3 - \sqrt{9-\varepsilon}, \sqrt{9+\varepsilon} - 3)$, larger by a factor tending to
$7/6$ as $\varepsilon \to 0$.

In [12]:
for eps in (1e-1, 1e-2, 1e-4):
    d_cert = min(1.0, eps / 7)
    d_sharp = min(3 - np.sqrt(9 - eps), np.sqrt(9 + eps) - 3)
    xs = np.linspace(3 - d_cert, 3 + d_cert, 200001)
    worst = np.abs(xs ** 2 - 9).max()
    print(f"  eps = {eps:.0e}   delta = {d_cert:.3e}   sup error = {worst:.3e}"
          f"   sharp/cert = {d_sharp/d_cert:.6f}")
    assert worst < eps
print("limit of sharp/cert as eps -> 0 is 7/6 =", 7 / 6)

  eps = 1e-01   delta = 1.429e-02   sup error = 8.592e-02   sharp/cert = 1.163444
  eps = 1e-02   delta = 1.429e-03   sup error = 8.573e-03   sharp/cert = 1.166343
  eps = 1e-04   delta = 1.429e-05   sup error = 8.571e-05   sharp/cert = 1.166663
limit of sharp/cert as eps -> 0 is 7/6 = 1.1666666666666667


### Problem L1.3 — An $\varepsilon$-$\delta$ proof for a reciprocal

**Statement.** Prove from Definition 3.1 that $\lim_{x \to 2} \dfrac{1}{x} = \dfrac{1}{2}$.

**Intuition.** The error has $\lvert x \rvert$ in a denominator, so $x$ must first be kept away
from $0$.

**Solution.**

*Step 1 — factor.*

$$
\left\lvert \frac{1}{x} - \frac{1}{2} \right\rvert = \frac{\lvert x - 2 \rvert}{2 \lvert x \rvert} .
$$

*Step 2 — cap.* Agree that $\delta \le 1$. Then $1 \lt x \lt 3$, so $\lvert x \rvert \gt 1$ and
$1/\lvert x \rvert \lt 1$.

*Step 3 — solve.* Under the cap the error is below $\lvert x - 2 \rvert / 2$, so
$\lvert x - 2 \rvert \lt 2\varepsilon$ suffices.

$$
\boxed{\delta(\varepsilon) = \min(1, 2\varepsilon)}
$$

**Key takeaway.** Every rational function needs this two-stage argument: one cap to control the
denominator, one inequality to control the numerator.

In [13]:
for eps in (1e-1, 1e-3, 1e-6):
    d = min(1.0, 2 * eps)
    xs = np.linspace(2 - d, 2 + d, 200001)
    worst = np.abs(1 / xs - 0.5).max()
    print(f"  eps = {eps:.0e}   delta = {d:.3e}   sup |1/x - 1/2| = {worst:.3e}")
    assert worst < eps

  eps = 1e-01   delta = 2.000e-01   sup |1/x - 1/2| = 5.556e-02
  eps = 1e-03   delta = 2.000e-03   sup |1/x - 1/2| = 5.005e-04
  eps = 1e-06   delta = 2.000e-06   sup |1/x - 1/2| = 5.000e-07


### Problem L1.4 — A conjugate kills a $0/0$

**Statement.** Evaluate $\displaystyle \lim_{x \to 0} \frac{\sqrt{1+x} - \sqrt{1-x}}{x}$.

**Intuition.** Multiplying by the conjugate turns a difference of roots into a difference of their
squares, which cancels the offending factor $x$.

**Solution.**

*Step 1 — rationalize.*

$$
\frac{\sqrt{1+x} - \sqrt{1-x}}{x} \cdot \frac{\sqrt{1+x} + \sqrt{1-x}}{\sqrt{1+x} + \sqrt{1-x}}
= \frac{(1+x) - (1-x)}{x \left( \sqrt{1+x} + \sqrt{1-x} \right)} .
$$

*Step 2 — cancel.* The numerator is $2x$, and $x \neq 0$ on the punctured neighbourhood, so the
expression equals $\dfrac{2}{\sqrt{1+x} + \sqrt{1-x}}$.

*Step 3 — evaluate.* The denominator is continuous at $0$ with value $2$, so Theorem 4.2 applies.

$$
\boxed{1}
$$

**Key takeaway.** Cancelling $x$ is legal precisely because the limit never inspects $x = 0$.

In [14]:
for x in (1e-2, 1e-4, 1e-6, 1e-8):
    naive = (np.sqrt(1 + x) - np.sqrt(1 - x)) / x
    stable = 2.0 / (np.sqrt(1 + x) + np.sqrt(1 - x))
    print(f"  x = {x:.0e}   naive = {naive:.14f}   rationalized = {stable:.14f}")
    assert abs(stable - 1.0) < 1e-3
print("limit = 1; the rationalized form also removes the cancellation the naive one suffers")

  x = 1e-02   naive = 1.00001250054690   rationalized = 1.00001250054691
  x = 1e-04   naive = 1.00000000124889   rationalized = 1.00000000125000
  x = 1e-06   naive = 1.00000000002876   rationalized = 1.00000000000013
  x = 1e-08   naive = 1.00000000502476   rationalized = 1.00000000000000
limit = 1; the rationalized form also removes the cancellation the naive one suffers


### Problem L1.5 — A trigonometric limit built from the two fundamental ones

**Statement.** Evaluate $\displaystyle \lim_{x \to 0} \frac{1 - \cos 3x}{x \sin 2x}$.

**Intuition.** Rewrite everything in terms of $\dfrac{1-\cos u}{u^{2}}$ and $\dfrac{\sin u}{u}$,
whose limits Theorem 4.10 supplies.

**Solution.**

*Step 1 — normalize each factor.* For $x \neq 0$,

$$
\frac{1 - \cos 3x}{x \sin 2x}
= \frac{1 - \cos 3x}{(3x)^{2}} \cdot \frac{(3x)^{2}}{x \sin 2x} .
$$

*Step 2 — normalize the sine.* $x \sin 2x = 2x^{2} \cdot \dfrac{\sin 2x}{2x}$, so the second
factor is $\dfrac{9x^{2}}{2x^{2}} \cdot \left( \dfrac{\sin 2x}{2x} \right)^{-1}$.

*Step 3 — take limits.* By Theorem 4.10 the first factor tends to $\tfrac{1}{2}$ and
$\dfrac{\sin 2x}{2x} \to 1$; Theorem 4.2 handles the product and the quotient, the latter because
the denominator limit $1$ is non-zero.

$$
\boxed{\lim_{x \to 0} \frac{1 - \cos 3x}{x \sin 2x} = \frac{1}{2} \cdot \frac{9}{2} = \frac{9}{4}}
$$

**Key takeaway.** Every elementary trigonometric limit reduces to the two numbers $1$ and
$\tfrac{1}{2}$ of Theorem 4.10 plus the algebra of Theorem 4.2.

In [15]:
for x in (1e-1, 1e-2, 1e-3, 1e-4):
    val = (1 - np.cos(3 * x)) / (x * np.sin(2 * x))
    print(f"  x = {x:.0e}   value = {val:.12f}   9/4 = {9/4:.12f}   error = {abs(val - 9/4):.3e}")
assert abs((1 - np.cos(3e-4)) / (1e-4 * np.sin(2e-4)) - 9 / 4) < 1e-5

  x = 1e-01   value = 2.248133151486   9/4 = 2.250000000000   error = 1.867e-03
  x = 1e-02   value = 2.249981250812   9/4 = 2.250000000000   error = 1.875e-05
  x = 1e-03   value = 2.249999812498   9/4 = 2.250000000000   error = 1.875e-07
  x = 1e-04   value = 2.249999995775   9/4 = 2.250000000000   error = 4.225e-09


### Problem L1.6 — An indeterminate form $1^{\infty}$

**Statement.** Evaluate $\displaystyle \lim_{x \to 0} (1 + 4x)^{\frac{1}{2x}}$.

**Intuition.** Every $1^{\infty}$ limit is the exponential base limit in disguise; substitute so
that the exponent is the reciprocal of the increment.

**Solution.**

*Step 1 — substitute.* Put $u = 4x$, so $u \to 0$ and $\dfrac{1}{2x} = \dfrac{2}{u}$.

*Step 2 — rewrite.*

$$
(1 + 4x)^{\frac{1}{2x}} = (1 + u)^{\frac{2}{u}} = \left[ (1+u)^{\frac{1}{u}} \right]^{2} .
$$

*Step 3 — pass to the limit.* $(1+u)^{1/u} \to e$, and $y \mapsto y^{2}$ is continuous at $e$, so
Proposition 4.13 lets the square move outside the limit.

$$
\boxed{e^{2}}
$$

**Key takeaway.** The substitution is legal because $u = 4x$ is continuous and non-zero on the
punctured neighbourhood; the squaring step is legal because Proposition 4.13 requires continuity
of the *outer* function, which $y^{2}$ has everywhere.

In [16]:
for x in (1e-2, 1e-4, 1e-6, 1e-8):
    print(f"  x = {x:.0e}   (1+4x)^(1/(2x)) = {(1 + 4*x) ** (1/(2*x)):.12f}")
print(f"  e^2 = {np.e ** 2:.12f}")
assert abs((1 + 4e-8) ** (1 / 2e-8) - np.e ** 2) < 1e-5

  x = 1e-02   (1+4x)^(1/(2x)) = 7.106683346278
  x = 1e-04   (1+4x)^(1/(2x)) = 7.386101855150
  x = 1e-06   (1+4x)^(1/(2x)) = 7.389026542449
  x = 1e-08   (1+4x)^(1/(2x)) = 7.389055795590
  e^2 = 7.389056098931


### Problem L1.7 — A squeeze on a wildly oscillating product

**Statement.** Compute $\displaystyle \lim_{x \to 0} x^{2} \sin \frac{1}{x}$ and give an explicit
$\delta(\varepsilon)$.

**Intuition.** A bounded factor times an infinitesimal factor is infinitesimal, no matter how
badly the bounded factor behaves.

**Solution.**

*Step 1 — sandwich.* $-1 \le \sin(1/x) \le 1$ for $x \neq 0$, and $x^{2} \ge 0$, so

$$
-x^{2} \ \le \ x^{2} \sin\frac{1}{x} \ \le \ x^{2} .
$$

*Step 2 — the envelopes agree.* Both tend to $0$, so Theorem 4.3 gives the limit $0$.

*Step 3 — read off $\delta$.* $\lvert x^{2}\sin(1/x) \rvert \le x^{2} \lt \varepsilon$ whenever
$\lvert x \rvert \lt \sqrt{\varepsilon}$.

$$
\boxed{\lim_{x \to 0} x^{2}\sin\frac{1}{x} = 0, \qquad \delta(\varepsilon) = \sqrt{\varepsilon}}
$$

**Key takeaway.** Theorem 4.2 is unusable here because $\sin(1/x)$ has no limit; Theorem 4.3
needs no limit for the middle function at all.

In [17]:
for eps in (1e-2, 1e-4, 1e-6):
    d = np.sqrt(eps)
    xs = np.linspace(-d, d, 400001)
    xs = xs[xs != 0.0]
    worst = np.abs(xs ** 2 * np.sin(1 / xs)).max()
    print(f"  eps = {eps:.0e}   delta = sqrt(eps) = {d:.3e}   sup |f| = {worst:.3e}")
    assert worst < eps

  eps = 1e-02   delta = sqrt(eps) = 1.000e-01   sup |f| = 8.411e-03


  eps = 1e-04   delta = sqrt(eps) = 1.000e-02   sup |f| = 9.594e-05


  eps = 1e-06   delta = sqrt(eps) = 1.000e-03   sup |f| = 9.988e-07


### Problem L1.8 — An $\infty - \infty$ limit

**Statement.** Evaluate $\displaystyle \lim_{x \to \infty} \left( \sqrt{x^{2} + 5x} - x \right)$.

**Intuition.** Two quantities both racing to infinity; rationalizing turns the race into a
bounded ratio.

**Solution.**

*Step 1 — rationalize.*

$$
\sqrt{x^{2}+5x} - x = \frac{(x^{2}+5x) - x^{2}}{\sqrt{x^{2}+5x} + x} = \frac{5x}{\sqrt{x^{2}+5x} + x} .
$$

*Step 2 — divide by $x \gt 0$.*

$$
= \frac{5}{\sqrt{1 + 5/x} + 1} .
$$

*Step 3 — take the limit.* $5/x \to 0$ and $\sqrt{\cdot}$ is continuous at $1$, so
Proposition 4.13 gives the denominator limit $2$.

$$
\boxed{\frac{5}{2}}
$$

**Key takeaway.** In general $\sqrt{x^{2}+bx} - x \to b/2$: the square root is $x + b/2 + O(1/x)$,
which is the first term of its asymptotic expansion.

In [18]:
for x in (1e2, 1e4, 1e6, 1e8):
    print(f"  x = {x:.0e}   sqrt(x^2+5x) - x = {np.sqrt(x**2 + 5*x) - x:.12f}"
          f"   rationalized = {5 / (np.sqrt(1 + 5/x) + 1):.12f}")
assert abs(5 / (np.sqrt(1 + 5 / 1e8) + 1) - 2.5) < 1e-6

  x = 1e+02   sqrt(x^2+5x) - x = 2.469507659596   rationalized = 2.469507659596
  x = 1e+04   sqrt(x^2+5x) - x = 2.499687578100   rationalized = 2.499687578101
  x = 1e+06   sqrt(x^2+5x) - x = 2.499996875064   rationalized = 2.499996875008
  x = 1e+08   sqrt(x^2+5x) - x = 2.499999970198   rationalized = 2.499999968750


### Problem L1.9 — A root located by the Intermediate Value Theorem

**Statement.** Show that $f(x) = x^{5} - 3x - 1$ has a root in $[1, 2]$, and locate it to
$10^{-10}$.

**Intuition.** A polynomial is continuous, and a sign change across a closed interval cannot
happen without crossing zero.

**Solution.**

*Step 1 — continuity.* Polynomials are continuous on $\mathbb{R}$ by repeated use of Theorem 4.2,
so $f$ is continuous on $[1,2]$.

*Step 2 — sign change.* $f(1) = 1 - 3 - 1 = -3$ and $f(2) = 32 - 6 - 1 = 25$, so $d = 0$ lies
strictly between them.

*Step 3 — apply Theorem 4.5.* There is $c \in (1,2)$ with $f(c) = 0$.

*Step 4 — locate.* Halving the bracket $k$ times leaves width $2^{-k}$, so
$k = \lceil \log_{2} 10^{10} \rceil = 34$ steps suffice.

$$
\boxed{c \approx 1.3887919844, \ \text{ existence by Theorem 4.5, } \ 34 \ \text{bisection steps for } 10^{-10}}
$$

**Key takeaway.** Theorem 4.5 gives existence and nothing else; the digits come from the halving
argument of Example 6.5, whose rate is exactly $\tfrac{1}{2}$ per step.

In [19]:
from scipy.optimize import brentq


def poly5(t):
    return t ** 5 - 3 * t - 1


lo, hi, steps = 1.0, 2.0, 0
while hi - lo > 1e-10:
    mid = 0.5 * (lo + hi)
    if poly5(lo) * poly5(mid) < 0:
        hi = mid
    else:
        lo = mid
    steps += 1
root = 0.5 * (lo + hi)
print(f"f(1) = {poly5(1.0):+.1f}    f(2) = {poly5(2.0):+.1f}")
print(f"bisection root  = {root:.12f}   steps = {steps}   predicted = {int(np.ceil(np.log2(1e10)))}")
print(f"scipy brentq    = {brentq(poly5, 1.0, 2.0, xtol=1e-15, rtol=8.9e-16):.12f}")
assert steps == int(np.ceil(np.log2(1e10)))
assert abs(root - brentq(poly5, 1.0, 2.0, xtol=1e-15, rtol=8.9e-16)) < 1e-10

f(1) = -3.0    f(2) = +25.0
bisection root  = 1.388791984384   steps = 34   predicted = 34
scipy brentq    = 1.388791984407


### Problem L1.10 — Choosing a constant to make a piecewise function continuous

**Statement.** Find $c$ so that

$$
f(x) = \begin{cases} c x^{2} + 2x & x \lt 2 \\ x^{3} - c x & x \ge 2 \end{cases}
$$

is continuous on $\mathbb{R}$.

**Intuition.** Each branch is a polynomial, hence continuous on its own side; only the seam can
fail, and it fails exactly when the two one-sided limits disagree.

**Solution.**

*Step 1 — left limit.* $\lim_{x \to 2^{-}} f(x) = 4c + 4$.

*Step 2 — right limit and value.* $\lim_{x \to 2^{+}} f(x) = f(2) = 8 - 2c$.

*Step 3 — match.* Definition 3.4 needs both to equal $f(2)$, so

$$
4c + 4 = 8 - 2c \implies 6c = 4 .
$$

$$
\boxed{c = \frac{2}{3}}
$$

**Key takeaway.** Continuity of a piecewise formula is a finite set of scalar equations, one per
seam; away from the seams it is inherited from the branches.

In [20]:
c = 2 / 3


def f_seam(x, c=c):
    return np.where(x < 2.0, c * x ** 2 + 2 * x, x ** 3 - c * x)


for h in (1e-6, 1e-9, 1e-12):
    left, right = f_seam(2 - h), f_seam(2 + h)
    print(f"  h = {h:.0e}   f(2-h) = {left:.12f}   f(2+h) = {right:.12f}   gap = {abs(left-right):.3e}")
print(f"c = {c:.10f}    f(2) = {f_seam(2.0):.12f}")
assert abs(f_seam(2 - 1e-12) - f_seam(2 + 1e-12)) < 1e-9
bad = f_seam(2 - 1e-9, c=1.0) - f_seam(2 + 1e-9, c=1.0)
print(f"with c = 1 instead, the seam gap is {bad:+.6f}: a jump discontinuity")
assert abs(bad) > 1e-3

  h = 1e-06   f(2-h) = 6.666662000001   f(2+h) = 6.666678000006   gap = 1.600e-05
  h = 1e-09   f(2-h) = 6.666666662000   f(2+h) = 6.666666678000   gap = 1.600e-08
  h = 1e-12   f(2-h) = 6.666666666662   f(2+h) = 6.666666666678   gap = 1.600e-11
c = 0.6666666667    f(2) = 6.666666666667
with c = 1 instead, the seam gap is +2.000000: a jump discontinuity


### Problem L1.11 — A limit from the logarithmic expansion

**Statement.** Evaluate $\displaystyle \lim_{x \to 0} \frac{\ln(1+x) - x}{x^{2}}$.

**Intuition.** Subtracting $x$ removes the linear term of $\ln(1+x)$, leaving the quadratic one.

**Solution.**

*Step 1 — expand.* $\ln(1+x) = x - \tfrac{1}{2}x^{2} + o(x^{2})$ as $x \to 0$, in the sense of
Definition 3.8.

*Step 2 — subtract.* $\ln(1+x) - x = -\tfrac{1}{2}x^{2} + o(x^{2})$.

*Step 3 — divide.* $\dfrac{\ln(1+x) - x}{x^{2}} = -\tfrac{1}{2} + o(1) \to -\tfrac{1}{2}$.

$$
\boxed{-\frac{1}{2}}
$$

**Key takeaway.** Asymptotic notation is doing real work here: $o(x^{2})/x^{2} \to 0$ *is* the
definition of $o$, so step 3 is an application of Definition 3.8, not an approximation.

In [21]:
print("      x        (log1p(x) - x)/x^2      series -1/2 + x/3")
for x in (1e-1, 1e-2, 1e-3, 1e-4, 1e-5):
    val = (np.log1p(x) - x) / x ** 2
    ser = -0.5 + x / 3
    print(f"  {x:7.0e}   {val:.12f}          {ser:.12f}")
    assert abs(val - ser) < 5e-2 * x + 1e-6
assert abs((np.log1p(1e-5) - 1e-5) / 1e-10 + 0.5) < 1e-4

      x        (log1p(x) - x)/x^2      series -1/2 + x/3
    1e-01   -0.468982019568          -0.466666666667
    1e-02   -0.496691468319          -0.496666666667
    1e-03   -0.499666916467          -0.499666666667
    1e-04   -0.499966669167          -0.499966666667
    1e-05   -0.499996666689          -0.499996666667


### Problem L1.12 — The sequential criterion refutes a limit

**Statement.** Use Theorem 4.4 to prove that $\lim_{x \to 0} \cos(1/x)$ does not exist.

**Intuition.** Two sequences approaching $0$ can be steered onto two different values of the
cosine, and the criterion says every sequence must give the same answer.

**Solution.**

*Step 1 — build two sequences.* For $n \ge 1$ put

$$
x_{n} = \frac{1}{2\pi n}, \qquad y_{n} = \frac{1}{2\pi n + \pi} .
$$

Both are non-zero and both tend to $0$.

*Step 2 — evaluate.* $\cos(1/x_{n}) = \cos(2\pi n) = 1$ and
$\cos(1/y_{n}) = \cos(2\pi n + \pi) = -1$.

*Step 3 — apply Theorem 4.4.* If the limit were $L$, the criterion would force $L = 1$ from the
first sequence and $L = -1$ from the second.

$$
\boxed{\lim_{x \to 0} \cos\frac{1}{x} \ \text{does not exist}; \ \liminf = -1 \lt 1 = \limsup}
$$

**Key takeaway.** The $(\Leftarrow)$ direction of Theorem 4.4 is what makes this a proof: one
sequence proves nothing, two disagreeing sequences settle the matter.

In [22]:
for n in (1, 2, 5, 50):
    xn = 1.0 / (2 * np.pi * n)
    yn = 1.0 / (2 * np.pi * n + np.pi)
    print(f"  n = {n:3d}   x_n = {xn:.10f}  cos(1/x_n) = {np.cos(1/xn):+.12f}"
          f"   y_n = {yn:.10f}  cos(1/y_n) = {np.cos(1/yn):+.12f}")
    assert abs(np.cos(1 / xn) - 1.0) < 1e-12
    assert abs(np.cos(1 / yn) + 1.0) < 1e-12
print("two sequences tending to 0 give images pinned at +1 and -1: no limit")

  n =   1   x_n = 0.1591549431  cos(1/x_n) = +1.000000000000   y_n = 0.1061032954  cos(1/y_n) = -1.000000000000
  n =   2   x_n = 0.0795774715  cos(1/x_n) = +1.000000000000   y_n = 0.0636619772  cos(1/y_n) = -1.000000000000
  n =   5   x_n = 0.0318309886  cos(1/x_n) = +1.000000000000   y_n = 0.0289372624  cos(1/y_n) = -1.000000000000
  n =  50   x_n = 0.0031830989  cos(1/x_n) = +1.000000000000   y_n = 0.0031515830  cos(1/y_n) = -1.000000000000
two sequences tending to 0 give images pinned at +1 and -1: no limit


### Problem L1.13 — Lipschitz implies uniformly continuous

**Statement.** Show that $\sin$ is Lipschitz on $\mathbb{R}$ with constant $K = 1$, deduce that it
is uniformly continuous there, and explain why the same argument fails for $x^{2}$ on
$[0,\infty)$ but succeeds on every $[0, R]$.

**Intuition.** A Lipschitz bound is a $\delta$ that scales linearly with $\varepsilon$ and never
mentions the base point.

**Solution.**

*Step 1 — the Lipschitz bound for $\sin$.* The sum-to-product identity gives

$$
\lvert \sin x - \sin y \rvert = 2 \left\lvert \sin \frac{x-y}{2} \right\rvert \left\lvert \cos \frac{x+y}{2} \right\rvert \le 2 \left\lvert \frac{x-y}{2} \right\rvert = \lvert x - y \rvert,
$$

using $\lvert \sin t \rvert \le \lvert t \rvert$ — itself the inequality $\sin t \lt t$ of
Proof 5.10 extended by oddness — and $\lvert \cos \rvert \le 1$.

*Step 2 — uniform continuity.* By Proof 5.11 the choice $\delta = \varepsilon / K = \varepsilon$
serves every pair at once, which is Definition 3.7.

*Step 3 — why $x^{2}$ differs.* On $[0,R]$,
$\lvert x^{2} - y^{2} \rvert = \lvert x-y \rvert \lvert x+y \rvert \le 2R \lvert x - y \rvert$, so
$K = 2R$ works and $\delta = \varepsilon/(2R)$. On $[0,\infty)$ the factor $\lvert x + y \rvert$
is unbounded, so no finite $K$ exists — consistent with Theorem 4.8, which only promises
uniformity on a compact interval.

$$
\boxed{\lvert \sin x - \sin y \rvert \le \lvert x - y \rvert \ \Rightarrow \ \delta(\varepsilon) = \varepsilon \ \text{on all of } \mathbb{R}; \quad x^{2} \ \text{is } 2R\text{-Lipschitz on } [0,R] \ \text{only}}
$$

**Key takeaway.** Lipschitz is strictly stronger than uniform continuity (Proposition 4.11), and
strictly stronger than what Theorem 4.8 delivers: Heine-Cantor gives a $\delta$, never a linear
one.

In [23]:
u = rng.uniform(-50.0, 50.0, 200000)
v = rng.uniform(-50.0, 50.0, 200000)
ratio = np.abs(np.sin(u) - np.sin(v)) / np.abs(u - v)
print(f"max |sin u - sin v| / |u - v| over 200000 random pairs : {ratio.max():.12f}   (bound 1)")
assert ratio.max() <= 1.0 + 1e-12

R = 5.0
uu = rng.uniform(0.0, R, 200000)
vv = rng.uniform(0.0, R, 200000)
ratio2 = np.abs(uu ** 2 - vv ** 2) / np.abs(uu - vv)
print(f"max |u^2 - v^2| / |u - v| on [0, {R:.0f}]                  : {ratio2.max():.6f}   (bound 2R = {2*R:.0f})")
assert ratio2.max() <= 2 * R + 1e-9
for x in (10.0, 1e3, 1e6):
    print(f"  on [0, inf): the ratio at (x, x + 1) is {abs((x+1)**2 - x**2):.1f} -> unbounded")

max |sin u - sin v| / |u - v| over 200000 random pairs : 0.999834690668   (bound 1)
max |u^2 - v^2| / |u - v| on [0, 5]                  : 9.977356   (bound 2R = 10)
  on [0, inf): the ratio at (x, x + 1) is 21.0 -> unbounded
  on [0, inf): the ratio at (x, x + 1) is 2001.0 -> unbounded
  on [0, inf): the ratio at (x, x + 1) is 2000001.0 -> unbounded


## L2 — Applications (AI/ML and Physics)

### Problem L2.1 — Softmax temperature: the two endpoints are limits

**Statement.** For logits $z = (3.0, 1.0, 0.5)$ and temperature $T \gt 0$, let

$$
p_{i}(T) = \frac{e^{z_{i}/T}}{\sum_{j=1}^{3} e^{z_{j}/T}} .
$$

Compute $\lim_{T \to 0^{+}} p_{1}(T)$ and $\lim_{T \to \infty} p_{1}(T)$.

**Intuition.** Low temperature amplifies the gaps between logits; high temperature erases them.

**Solution.**

*Step 1 — shift to the maximum.* The maximum is $z_{1} = 3.0$. Dividing numerator and denominator
by $e^{z_{1}/T}$,

$$
p_{1}(T) = \frac{1}{1 + e^{-2.0/T} + e^{-2.5/T}} .
$$

*Step 2 — the cold limit.* Both exponents tend to $-\infty$ as $T \to 0^{+}$, so both terms tend
to $0$ and the denominator tends to $1 \neq 0$; Theorem 4.2 gives $p_{1} \to 1$.

*Step 3 — the hot limit.* $z_{i}/T \to 0$ and $\exp$ is continuous at $0$, so by
Proposition 4.13 every $e^{z_{i}/T} \to 1$ and the ratio tends to $1/3$.

$$
\boxed{\lim_{T \to 0^{+}} p_{1}(T) = 1, \qquad \lim_{T \to \infty} p_{1}(T) = \frac{1}{K} = \frac{1}{3}}
$$

**Key takeaway.** Temperature is a homotopy from $\operatorname{argmax}$ to the uniform
distribution, and both ends are limits: at $T = 0$ the formula is undefined.

In [24]:
z = np.array([3.0, 1.0, 0.5])


def softmax_T(z, T):
    e = np.exp((z - z.max()) / T)
    return e / e.sum()


for T in (1e3, 10.0, 1.0, 0.1, 1e-2, 1e-3):
    p = softmax_T(z, T)
    print(f"  T = {T:8.3g}   p1 = {p[0]:.12f}   p = {np.array2string(p, precision=6)}")
assert abs(softmax_T(z, 1e-3)[0] - 1.0) < 1e-12
assert abs(softmax_T(z, 1e6)[0] - 1 / 3) < 1e-5

  T =    1e+03   p1 = 0.333833513743   p = [0.333834 0.333167 0.333   ]
  T =       10   p1 = 0.384980889003   p = [0.384981 0.315196 0.299823]
  T =        1   p1 = 0.821409019465   p = [0.821409 0.111166 0.067425]
  T =      0.1   p1 = 0.999999997925   p = [1. 0. 0.]
  T =     0.01   p1 = 1.000000000000   p = [1. 0. 0.]
  T =    0.001   p1 = 1.000000000000   p = [1. 0. 0.]


### Problem L2.2 — The log-sum-exp window

**Statement.** Prove the two-sided bound $m \le \operatorname{LSE}(x) \le m + \ln n$ for
$\operatorname{LSE}(x) = \ln \sum_{i=1}^{n} e^{x_{i}}$ with $m = \max_{i} x_{i}$, and deduce
$\tfrac{1}{k}\operatorname{LSE}(kx) \to m$ as $k \to \infty$.

**Intuition.** The sum lies between its largest term and $n$ copies of it; the logarithm turns
that factor of $n$ into an additive $\ln n$.

**Solution.**

*Step 1 — bound the sum.* Every term is at most $e^{m}$ and one term equals $e^{m}$, so

$$
e^{m} \ \le \ \sum_{i=1}^{n} e^{x_{i}} \ \le \ n e^{m} .
$$

*Step 2 — take logarithms.* $\ln$ is increasing, so the inequalities survive:
$m \le \operatorname{LSE}(x) \le m + \ln n$.

*Step 3 — rescale.* Replacing $x$ by $kx$ multiplies $m$ by $k$, so

$$
m \ \le \ \frac{1}{k}\operatorname{LSE}(kx) \ \le \ m + \frac{\ln n}{k} .
$$

*Step 4 — squeeze.* Both envelopes tend to $m$ as $k \to \infty$, so Theorem 4.3 gives the limit.

$$
\boxed{m \le \operatorname{LSE}(x) \le m + \ln n, \qquad \lim_{k \to \infty} \frac{1}{k}\operatorname{LSE}(kx) = \max_{i} x_{i}}
$$

**Key takeaway.** $\operatorname{LSE}$ is a smooth maximum with an error at most $\ln n$,
*uniformly in $x$* — which is why temperature-scaled attention and soft-min objectives behave
predictably no matter how large the logits get.

In [25]:
from scipy.special import logsumexp


def lse_stable(x):
    m = np.max(x)
    return m + np.log(np.sum(np.exp(x - m)))


for v in (np.array([1.0, 2.0, 3.0]), np.zeros(4), np.array([700.0, 701.0, 699.0])):
    m, n = v.max(), v.size
    val = lse_stable(v)
    print(f"  x = {np.array2string(v, precision=1):<22} m = {m:8.3f}   LSE = {val:.10f}"
          f"   m + ln n = {m + np.log(n):.10f}   scipy = {logsumexp(v):.10f}")
    assert m - 1e-12 <= val <= m + np.log(n) + 1e-12
    assert abs(val - logsumexp(v)) < 1e-12

x0 = np.array([1.0, 2.0, 3.0])
for k in (1.0, 10.0, 100.0, 1000.0):
    print(f"  k = {k:7.1f}   LSE(kx)/k = {lse_stable(k * x0) / k:.12f}   (max = {x0.max():.1f})")
assert abs(lse_stable(1000.0 * x0) / 1000.0 - 3.0) < 1e-3

  x = [1. 2. 3.]             m =    3.000   LSE = 3.4076059644   m + ln n = 4.0986122887   scipy = 3.4076059644
  x = [0. 0. 0. 0.]          m =    0.000   LSE = 1.3862943611   m + ln n = 1.3862943611   scipy = 1.3862943611
  x = [700. 701. 699.]       m =  701.000   LSE = 701.4076059644   m + ln n = 702.0986122887   scipy = 701.4076059644
  k =     1.0   LSE(kx)/k = 3.407605964444   (max = 3.0)
  k =    10.0   LSE(kx)/k = 3.000004540096   (max = 3.0)
  k =   100.0   LSE(kx)/k = 3.000000000000   (max = 3.0)
  k =  1000.0   LSE(kx)/k = 3.000000000000   (max = 3.0)


### Problem L2.3 — The Newtonian limit of relativistic kinetic energy (physics)

**Statement.** With $\gamma(v) = (1 - v^{2}/c^{2})^{-1/2}$ and $E_{k} = (\gamma - 1) m_{0} c^{2}$,
prove that

$$
\lim_{v \to 0} \frac{E_{k}}{\tfrac{1}{2} m_{0} v^{2}} = 1 .
$$

**Intuition.** Relativity must reproduce Newton at low speed; the statement is an asymptotic
equivalence in the sense of Definition 3.8.

**Solution.**

*Step 1 — substitute.* Put $s = v^{2}/c^{2}$, so $s \to 0^{+}$ as $v \to 0$.

*Step 2 — expand.* $(1-s)^{-1/2} = 1 + \tfrac{1}{2}s + \tfrac{3}{8}s^{2} + O(s^{3})$.

*Step 3 — form the energy.*

$$
E_{k} = \left( \tfrac{1}{2}s + \tfrac{3}{8}s^{2} + O(s^{3}) \right) m_{0}c^{2} = \tfrac{1}{2} m_{0}v^{2} + \tfrac{3}{8}\frac{m_{0}v^{4}}{c^{2}} + O\!\left( \frac{v^{6}}{c^{4}} \right) .
$$

*Step 4 — divide.* The ratio is $1 + \tfrac{3}{4} v^{2}/c^{2} + O(v^{4}/c^{4}) \to 1$.

$$
\boxed{E_{k} \sim \tfrac{1}{2} m_{0} v^{2} \ \ (v \to 0), \quad \text{with relative error } \tfrac{3}{4}\frac{v^{2}}{c^{2}} + O\!\left(\frac{v^{4}}{c^{4}}\right)}
$$

**Key takeaway.** Asymptotic equivalence says nothing about $v$ comparable to $c$: at
$v = 0.9c$ the ratio is above $3$, so the Newtonian formula is not an approximation there, it is
simply wrong.

In [26]:
def gamma_minus_one(beta):
    """(1 - b^2)^(-1/2) - 1 written so that no cancellation occurs as b -> 0."""
    s = np.sqrt(1 - beta ** 2)
    return beta ** 2 / (s * (1 + s))


print("   v/c     naive gamma - 1        stable gamma - 1     E_k/(m v^2/2)   1 + (3/4)(v/c)^2")
for beta in (1e-4, 1e-3, 1e-2, 0.1, 0.5, 0.9):
    naive = 1.0 / np.sqrt(1 - beta ** 2) - 1.0
    stable = gamma_minus_one(beta)
    ratio = stable / (0.5 * beta ** 2)
    print(f"  {beta:6.4f}   {naive:.16e}  {stable:.16e}   {ratio:14.10f}   {1 + 0.75*beta**2:.10f}")
    if beta <= 1e-2:
        assert abs(ratio - (1 + 0.75 * beta ** 2)) < 10 * beta ** 4 + 1e-15
print("the naive column loses eight digits at v/c = 1e-4; the ratio itself still tends to 1")
assert abs(gamma_minus_one(1e-6) / (0.5e-12) - 1.0) < 1e-11

   v/c     naive gamma - 1        stable gamma - 1     E_k/(m v^2/2)   1 + (3/4)(v/c)^2
  0.0001   5.0000001916572501e-09  5.0000000375000007e-09     1.0000000075   1.0000000075
  0.0010   5.0000037488118210e-07  5.0000037500031241e-07     1.0000007500   1.0000007500
  0.0100   5.0003750312610507e-05  5.0003750312527348e-05     1.0000750063   1.0000750000
  0.1000   5.0378152592120973e-03  5.0378152592120765e-03     1.0075630518   1.0075000000
  0.5000   1.5470053837925168e-01  1.5470053837925155e-01     1.2376043070   1.1875000000
  0.9000   1.2941573387056180e+00  1.2941573387056182e+00     3.1954502190   1.6075000000
the naive column loses eight digits at v/c = 1e-4; the ratio itself still tends to 1


### Problem L2.4 — The two tails of GELU

**Statement.** For $f(x) = x\Phi(x)$ with $\Phi$ the standard normal CDF, find
$\lim_{x \to +\infty} f(x)/x$ and $\lim_{x \to -\infty} f(x)$.

**Intuition.** On the right the gate opens fully and GELU becomes the identity; on the left the
Gaussian tail beats the linear factor.

**Solution.**

*Step 1 — the right tail.* $\Phi(x) \to 1$ as $x \to +\infty$, so $f(x)/x = \Phi(x) \to 1$, that
is $f(x) \sim x$ in the sense of Definition 3.8.

*Step 2 — the left tail.* The Mills-ratio asymptotic gives
$\Phi(x) \sim \dfrac{1}{\sqrt{2\pi}\,\lvert x \rvert} e^{-x^{2}/2}$ as $x \to -\infty$, hence

$$
f(x) = x\Phi(x) \sim -\frac{1}{\sqrt{2\pi}} e^{-x^{2}/2} .
$$

*Step 3 — take the limit.* The exponential dominates every power, so $f(x) \to 0$, from below.

$$
\boxed{\lim_{x \to +\infty} \frac{f(x)}{x} = 1, \qquad \lim_{x \to -\infty} f(x) = 0^{-}}
$$

**Key takeaway.** GELU is asymptotically ReLU but is smooth everywhere, and its negative tail is
non-zero, which is what distinguishes it from a hard gate.

In [27]:
from scipy.special import ndtr


def gelu(x):
    return x * ndtr(x)


for x in (1.0, 5.0, 10.0, 50.0):
    print(f"  x = {x:6.1f}   gelu(x)/x = {gelu(x)/x:.14f}")
for x in (-2.0, -5.0, -8.0, -12.0):
    approx = -np.exp(-x ** 2 / 2) / np.sqrt(2 * np.pi)
    print(f"  x = {x:6.1f}   gelu(x) = {gelu(x):+.6e}   tail model = {approx:+.6e}"
          f"   ratio = {gelu(x)/approx:.6f}")
assert abs(gelu(50.0) / 50.0 - 1.0) < 1e-12
assert abs(gelu(-12.0)) < 1e-30
assert 0.9 < gelu(-12.0) / (-np.exp(-72.0) / np.sqrt(2 * np.pi)) < 1.1

  x =    1.0   gelu(x)/x = 0.84134474606854
  x =    5.0   gelu(x)/x = 0.99999971334843
  x =   10.0   gelu(x)/x = 1.00000000000000
  x =   50.0   gelu(x)/x = 1.00000000000000
  x =   -2.0   gelu(x) = -4.550026e-02   tail model = -5.399097e-02   ratio = 0.842738
  x =   -5.0   gelu(x) = -1.433258e-06   tail model = -1.486720e-06   ratio = 0.964041
  x =   -8.0   gelu(x) = -4.976768e-15   tail model = -5.052271e-15   ratio = 0.985056
  x =  -12.0   gelu(x) = -2.131779e-32   tail model = -2.146384e-32   ratio = 0.993195


### Problem L2.5 — Swish interpolates between a half-line and ReLU

**Statement.** For $f_{\beta}(x) = x \sigma(\beta x)$ with $\sigma(t) = (1 + e^{-t})^{-1}$,
compute $\lim_{\beta \to \infty} f_{\beta}(x)$ and $\lim_{\beta \to 0^{+}} f_{\beta}(x)$ at fixed
$x$.

**Intuition.** The sigmoid sharpens into a step as $\beta$ grows and flattens to $\tfrac{1}{2}$ as
$\beta$ shrinks.

**Solution.**

*Step 1 — large $\beta$, three cases.* For $x \gt 0$, $\beta x \to +\infty$ so
$\sigma(\beta x) \to 1$ and $f_{\beta}(x) \to x$. For $x \lt 0$, $\beta x \to -\infty$ so
$\sigma \to 0$ and $f_{\beta}(x) \to 0$. For $x = 0$, $f_{\beta}(0) = 0$ for every $\beta$.

*Step 2 — collect.* The three cases are exactly $\max(0,x)$.

*Step 3 — small $\beta$.* $\beta x \to 0$ and $\sigma$ is continuous at $0$ with
$\sigma(0) = \tfrac{1}{2}$, so Proposition 4.13 gives $f_{\beta}(x) \to x/2$.

$$
\boxed{\lim_{\beta \to \infty} f_{\beta}(x) = \operatorname{ReLU}(x), \qquad \lim_{\beta \to 0^{+}} f_{\beta}(x) = \frac{x}{2}}
$$

**Key takeaway.** The convergence to ReLU is pointwise but not uniform: near $x = 0$ the two
curves stay a distance $\Theta(1/\beta)$ apart, which is precisely the smoothing that makes Swish
differentiable where ReLU is not.

In [28]:
def swish(x, beta):
    return x / (1.0 + np.exp(-beta * x))


xs = np.array([-2.0, -0.5, 0.0, 0.5, 2.0])
print("  beta        swish at x =", xs)
for beta in (0.01, 1.0, 10.0, 100.0, 1e4):
    print(f"  {beta:8.2f}   {np.array2string(swish(xs, beta), precision=6)}")
print("  ReLU       ", np.array2string(np.maximum(xs, 0.0), precision=6))
print("  x/2        ", np.array2string(xs / 2, precision=6))
assert np.allclose(swish(xs, 1e4), np.maximum(xs, 0.0), atol=1e-6)
assert np.allclose(swish(xs, 1e-6), xs / 2, atol=1e-5)
gap = np.abs(swish(np.linspace(-1, 1, 4001), 50.0) - np.maximum(np.linspace(-1, 1, 4001), 0.0)).max()
print(f"  worst gap to ReLU at beta = 50 : {gap:.6f}   (model 0.693/beta = {np.log(2)/50:.6f})")

  beta        swish at x = [-2.  -0.5  0.   0.5  2. ]
      0.01   [-0.99     -0.249375  0.        0.250625  1.01    ]
      1.00   [-0.238406 -0.18877   0.        0.31123   1.761594]
     10.00   [-0.       -0.003346  0.        0.496654  2.      ]
    100.00   [-0.  -0.   0.   0.5  2. ]
  10000.00   [-0.  -0.   0.   0.5  2. ]
  ReLU        [0.  0.  0.  0.5 2. ]
  x/2         [-1.   -0.25  0.    0.25  1.  ]
  worst gap to ReLU at beta = 50 : 0.005569   (model 0.693/beta = 0.013863)


/tmp/ipykernel_186340/3799543695.py:2: RuntimeWarning: overflow encountered in exp
  return x / (1.0 + np.exp(-beta * x))


### Problem L2.6 — How many bisection steps buy a given accuracy

**Statement.** How many bisection steps are needed to locate a root in $[0,1]$ to absolute error
$10^{-6}$?

**Intuition.** Each step halves the bracket exactly, so the count is a base-two logarithm.

**Solution.**

*Step 1 — the width after $k$ steps.* Example 6.5 shows the bracket width is exactly
$(b-a)2^{-k} = 2^{-k}$.

*Step 2 — impose the tolerance.* $2^{-k} \le 10^{-6} \iff k \ge \log_{2} 10^{6}$.

*Step 3 — evaluate.* $\log_{2}10^{6} = 6 \log_{2} 10 = 6 \times 3.321928\ldots = 19.9316\ldots$

*Step 4 — round up.* $k$ is an integer.

$$
\boxed{k = \lceil \log_{2} 10^{6} \rceil = 20 \ \text{steps}}
$$

**Key takeaway.** Bisection buys $\log_{10} 2 \approx 0.301$ decimal digits per step, or
$3.32$ steps per digit — a rate that does not depend on the function at all, only on Theorem 4.5
holding at every step.

In [29]:
k_pred = int(np.ceil(np.log2(1e6)))
print(f"log2(1e6) = {np.log2(1e6):.6f}   ceil = {k_pred}")


def bisect_width(a, b, tol):
    lo, hi, it = a, b, 0
    while hi - lo > tol:
        mid = 0.5 * (lo + hi)
        if np.cos(lo) * np.cos(mid) < 0:      # root of cos in [0, 2]
            hi = mid
        else:
            lo = mid
        it += 1
    return it, 0.5 * (lo + hi)


steps, root = bisect_width(0.0, 1.0, 1e-6)
print(f"width 2^-19 = {2.0**-19:.3e}  >  1e-6 ;  width 2^-20 = {2.0**-20:.3e}  <=  1e-6")
steps_scaled, _ = bisect_width(1.0, 2.0, 1e-6)
print(f"measured steps on a unit bracket : {steps_scaled}   predicted : {k_pred}")
assert steps_scaled == k_pred
assert 2.0 ** -20 <= 1e-6 < 2.0 ** -19

log2(1e6) = 19.931569   ceil = 20
width 2^-19 = 1.907e-06  >  1e-6 ;  width 2^-20 = 9.537e-07  <=  1e-6
measured steps on a unit bracket : 20   predicted : 20


### Problem L2.7 — The optimal step of a forward difference

**Statement.** For $f = \sin$ at $x = 1$, find the step $h$ minimizing the total error of
$\dfrac{f(x+h) - f(x)}{h}$ in `float64`, and state the smallest achievable error.

**Intuition.** Shrinking $h$ reduces the truncation error and inflates the cancellation error;
the optimum balances the two.

**Solution.**

*Step 1 — the two error terms.* Taylor's theorem gives a truncation error
$\tfrac{h}{2}\lvert f''(x) \rvert$, and the standard model with unit roundoff
$u = \tfrac{1}{2}\varepsilon_{\mathrm{mach}} = 2^{-53}$ gives a rounding error of order
$u \lvert f(x) \rvert / h$:

$$
E(h) \ \approx \ \frac{u \lvert f(x) \rvert}{h} + \frac{h}{2} \lvert f''(x) \rvert .
$$

*Step 2 — minimize.* Setting $E'(h) = 0$ gives
$-u\lvert f \rvert / h^{2} + \tfrac{1}{2}\lvert f'' \rvert = 0$, so

$$
h_{\star} = \sqrt{\frac{2u \lvert f(x) \rvert}{\lvert f''(x) \rvert}} .
$$

*Step 3 — specialize.* For $f = \sin$, $f'' = -\sin$, so $\lvert f \rvert = \lvert f'' \rvert$ at
every $x$ and the ratio cancels:

$$
h_{\star} = \sqrt{2u} = \sqrt{\varepsilon_{\mathrm{mach}}} = 2^{-26} = 1.4901 \times 10^{-8} .
$$

*Step 4 — the resulting error.* $E(h_{\star}) = \lvert f(x) \rvert \sqrt{2u} = \sin(1)\sqrt{2u} \approx 1.25 \times 10^{-8}$,
a worst-case figure: measured errors are about half of it because rounding errors partly cancel.

$$
\boxed{h_{\star} = \sqrt{2u} = 2^{-26} \approx 1.4901 \times 10^{-8}, \qquad E(h_{\star}) \lesssim 1.25 \times 10^{-8}}
$$

**Key takeaway.** Half the digits are gone no matter what: a forward difference cannot beat
$O(\sqrt{u})$, which is why automatic differentiation and complex-step derivatives exist. The
central difference shifts the optimum to $u^{1/3}$ and the error to $O(u^{2/3})$.

In [30]:
h_star = np.sqrt(2 * UNIT_ROUNDOFF)
print(f"h* = sqrt(2u) = {h_star:.10e}   2^-26 = {2.0**-26:.10e}")
print(f"model error E(h*) = |f| sqrt(2u) = {np.sin(1.0) * h_star:.6e}")

base = np.linspace(0.9, 1.1, 400)          # average to suppress single-point roundoff noise
h_grid = np.logspace(-1, -15, 141)
mean_err = np.array([np.abs((np.sin(base + hh) - np.sin(base)) / hh - np.cos(base)).mean()
                     for hh in h_grid])
i_min = int(np.argmin(mean_err))
err_star = np.abs((np.sin(base + h_star) - np.sin(base)) / h_star - np.cos(base)).mean()
print(f"empirical minimizer on the grid : {h_grid[i_min]:.6e}   error {mean_err[i_min]:.6e}")
print(f"error at the predicted h*       : {err_star:.6e}   ratio {err_star/mean_err[i_min]:.4f}")
assert abs(h_star - 2.0 ** -26) < 1e-24
assert err_star < 1.5 * mean_err[i_min]

h* = sqrt(2u) = 1.4901161194e-08   2^-26 = 1.4901161194e-08
model error E(h*) = |f| sqrt(2u) = 1.253889e-08
empirical minimizer on the grid : 1.000000e-08   error 5.619091e-09
error at the predicted h*       : 6.110727e-09   ratio 1.0875


### Problem L2.8 — A thick barrier switches tunnelling off (physics)

**Statement.** For $E \lt V_{0}$ the transmission probability through a rectangular barrier of
width $L$ is

$$
T(E) = \left[ 1 + \frac{V_{0}^{2}\sinh^{2}(\kappa L)}{4E(V_{0}-E)} \right]^{-1},
\qquad \kappa = \frac{\sqrt{2m(V_{0}-E)}}{\hbar} .
$$

Compute $\lim_{L \to \infty} T(E)$ and the rate at which it vanishes.

**Intuition.** $\sinh$ grows exponentially, so the bracket blows up and its reciprocal collapses.

**Solution.**

*Step 1 — the growth of the sinh.*
$\sinh(\kappa L) = \tfrac{1}{2}(e^{\kappa L} - e^{-\kappa L}) \sim \tfrac{1}{2}e^{\kappa L}$, so
$\sinh^{2}(\kappa L) \sim \tfrac{1}{4}e^{2\kappa L} \to \infty$.

*Step 2 — the bracket.* The added term is a positive constant times $\sinh^{2}$, so the bracket
tends to $+\infty$ in the sense of Definition 3.3.

*Step 3 — invert.* $t \mapsto 1/t$ tends to $0$ as $t \to +\infty$, giving $T \to 0$.

*Step 4 — the rate.* Dividing, $T(E) \sim \dfrac{16E(V_{0}-E)}{V_{0}^{2}} e^{-2\kappa L}$, so the
decay is exponential in $L$ with rate $2\kappa$.

$$
\boxed{\lim_{L \to \infty} T(E) = 0, \qquad T(E) \sim \frac{16E(V_{0}-E)}{V_{0}^{2}}\, e^{-2\kappa L}}
$$

**Key takeaway.** The limit is $0$ but the *rate* is what physics uses: doubling the barrier width
squares the transmission, which is why tunnelling currents in a scanning tunnelling microscope
resolve single atomic steps.

In [31]:
V0, E = 1.0, 0.4
kappa = 1.0


def T_exact(L):
    return 1.0 / (1.0 + V0 ** 2 * np.sinh(kappa * L) ** 2 / (4 * E * (V0 - E)))


pref = 16 * E * (V0 - E) / V0 ** 2
print("   L      T(E) exact        prefactor * exp(-2 kappa L)     ratio")
for L in (1.0, 2.0, 5.0, 10.0, 20.0):
    print(f"  {L:5.1f}   {T_exact(L):.6e}     {pref*np.exp(-2*kappa*L):.6e}"
          f"        {T_exact(L)/(pref*np.exp(-2*kappa*L)):.8f}")
assert T_exact(50.0) < 1e-40
assert abs(T_exact(20.0) / (pref * np.exp(-40.0)) - 1.0) < 1e-8

   L      T(E) exact        prefactor * exp(-2 kappa L)     ratio
    1.0   4.100640e-01     5.196875e-01        0.78905887
    2.0   6.801701e-02     7.033205e-02        0.96708410
    5.0   1.743212e-04     1.743357e-04        0.99991647
   10.0   7.914830e-09     7.914830e-09        1.00000000
   20.0   1.631368e-17     1.631368e-17        1.00000000


### Problem L2.9 — Why sigmoid networks lose their gradient

**Statement.** Show $\max_{t} \sigma'(t) = \tfrac{1}{4}$ for $\sigma(t) = (1+e^{-t})^{-1}$, and
evaluate $\lim_{L \to \infty} \left( \max_{t} \sigma'(t) \right)^{L}$.

**Intuition.** The steepest a sigmoid ever gets is at its midpoint, and multiplying $L$ numbers
below $1$ drives the product to zero geometrically.

**Solution.**

*Step 1 — reparametrize.* $\sigma'(t) = \sigma(t)(1 - \sigma(t))$, so with $y = \sigma(t) \in (0,1)$
the quantity to maximize is $g(y) = y - y^{2}$.

*Step 2 — maximize.* $g$ is a downward parabola with vertex at $y = \tfrac{1}{2}$, where
$g = \tfrac{1}{4}$. The value $y = \tfrac{1}{2}$ is attained, at $t = 0$.

*Step 3 — compound over depth.* A chain of $L$ sigmoid layers contributes a product of $L$ such
factors, bounded by $4^{-L}$.

*Step 4 — the limit.* $4^{-L} \to 0$, and geometrically: $10$ layers already give $10^{-6}$.

$$
\boxed{\max_{t}\sigma'(t) = \sigma'(0) = \frac{1}{4}, \qquad \lim_{L \to \infty} \left(\tfrac{1}{4}\right)^{L} = 0}
$$

**Key takeaway.** The bound is on the *maximum*, so the true product is even smaller; this is the
quantitative form of the vanishing-gradient problem, and it is why ReLU (derivative $1$ on the
positive half-line) replaced the sigmoid in deep stacks.

In [32]:
t = np.linspace(-10, 10, 2000001)
sig = 1.0 / (1.0 + np.exp(-t))
dsig = sig * (1 - sig)
print(f"max sigma'(t) on the grid = {dsig.max():.14f}   at t = {t[int(np.argmax(dsig))]:+.6f}")
print(f"exact maximum 1/4         = {0.25:.14f}")
assert abs(dsig.max() - 0.25) < 1e-12
for L in (1, 5, 10, 20, 50):
    print(f"  depth L = {L:3d}   (1/4)^L = {0.25**L:.6e}")
assert 0.25 ** 50 < 1e-30

max sigma'(t) on the grid = 0.25000000000000   at t = +0.000000
exact maximum 1/4         = 0.25000000000000
  depth L =   1   (1/4)^L = 2.500000e-01
  depth L =   5   (1/4)^L = 9.765625e-04
  depth L =  10   (1/4)^L = 9.536743e-07
  depth L =  20   (1/4)^L = 9.094947e-13
  depth L =  50   (1/4)^L = 7.888609e-31


### Problem L2.10 — Terminal velocity and free fall as two limits (physics)

**Statement.** A body falling with linear drag has
$v(t) = \dfrac{m g}{b}\left( 1 - e^{-bt/m} \right)$. Compute
$v_{\mathrm{term}} = \lim_{t \to \infty} v(t)$ and $\lim_{t \to 0^{+}} v(t)/t$.

**Intuition.** After a long time drag balances gravity; immediately after release there is no
speed yet, so no drag, and the motion is free fall.

**Solution.**

*Step 1 — the long-time limit.* $bt/m \to \infty$, so $e^{-bt/m} \to 0$ and
$v(t) \to mg/b$.

*Step 2 — the short-time expansion.* $e^{-s} = 1 - s + \tfrac{1}{2}s^{2} + O(s^{3})$ with
$s = bt/m$ gives

$$
v(t) = \frac{mg}{b}\left( \frac{bt}{m} - \frac{1}{2}\frac{b^{2}t^{2}}{m^{2}} + O(t^{3}) \right) = g t - \frac{gb}{2m} t^{2} + O(t^{3}) .
$$

*Step 3 — divide.* $v(t)/t = g - \dfrac{gb}{2m}t + O(t^{2}) \to g$.

$$
\boxed{v_{\mathrm{term}} = \frac{mg}{b}, \qquad \lim_{t \to 0^{+}} \frac{v(t)}{t} = g}
$$

**Key takeaway.** The same formula holds two different physical regimes, each recovered by a
different limit; the time scale separating them is $\tau = m/b$, the value at which
$v(\tau) = (1 - e^{-1}) v_{\mathrm{term}} \approx 0.632\, v_{\mathrm{term}}$.

In [33]:
m_mass, b_drag, g_acc = 2.0, 0.5, 9.81


def v_drag(t):
    return (m_mass * g_acc / b_drag) * (1 - np.exp(-b_drag * t / m_mass))


v_term = m_mass * g_acc / b_drag
tau = m_mass / b_drag
print(f"v_term = m g / b = {v_term:.6f} m/s     tau = m/b = {tau:.3f} s")
for t in (1e-6, 1e-3, 1.0, tau, 10 * tau, 100 * tau):
    print(f"  t = {t:10.4f}   v = {v_drag(t):12.6f}   v/t = {v_drag(t)/t:12.6f}   g = {g_acc}")
assert abs(v_drag(1e-6) / 1e-6 - g_acc) < 1e-4
assert abs(v_drag(100 * tau) - v_term) < 1e-12
assert abs(v_drag(tau) / v_term - (1 - np.exp(-1.0))) < 1e-12

v_term = m g / b = 39.240000 m/s     tau = m/b = 4.000 s
  t =     0.0000   v =     0.000010   v/t =     9.809999   g = 9.81
  t =     0.0010   v =     0.009809   v/t =     9.808774   g = 9.81
  t =     1.0000   v =     8.679857   v/t =     8.679857   g = 9.81
  t =     4.0000   v =    24.804411   v/t =     6.201103   g = 9.81
  t =    40.0000   v =    39.238219   v/t =     0.980955   g = 9.81
  t =   400.0000   v =    39.240000   v/t =     0.098100   g = 9.81


## L3 — Challenge Proofs

### Problem L3.1 — Thomae's function: continuous exactly at the irrationals

**Statement.** Define $f : (0,1) \to \mathbb{R}$ by $f(x) = 1/q$ when $x = p/q$ in lowest terms
with $p, q \in \mathbb{N}$, and $f(x) = 0$ when $x$ is irrational. Prove that $f$ is continuous at
every irrational point and discontinuous at every rational point.

**Intuition.** Near any point, the rationals with small denominator are sparse, so all the *large*
values of $f$ can be dodged by choosing $\delta$ small enough — but only if the centre itself is
not one of them.

**Solution.**

*Step 1 — discontinuity at a rational $a = p/q$.* Here $f(a) = 1/q \gt 0$. Every punctured
neighbourhood of $a$ contains irrationals, at which $f = 0$. Taking
$\varepsilon = 1/(2q)$, no $\delta$ can work, so Definition 3.4 fails.

*Step 2 — continuity at an irrational $a$, the finite set.* Here $f(a) = 0$. Given
$\varepsilon \gt 0$, pick $N$ with $1/N \lt \varepsilon$. The set

$$
F = \left\lbrace \frac{p}{q} \in (0,1) : q \le N \right\rbrace
$$

is **finite**: there are at most $q-1$ choices of $p$ for each $q \le N$.

*Step 3 — the certificate.* Since $a$ is irrational, $a \notin F$, and $F$ is finite, so

$$
\delta = \min_{r \in F} \lvert a - r \rvert \ \gt \ 0 .
$$

*Step 4 — verify.* Let $0 \lt \lvert x - a \rvert \lt \delta$. If $x$ is irrational then
$f(x) = 0$. If $x = p/q$ in lowest terms then $x \notin F$ by the choice of $\delta$, so $q \gt N$
and $f(x) = 1/q \lt 1/N \lt \varepsilon$. Either way
$\lvert f(x) - f(a) \rvert \lt \varepsilon$.

$$
\boxed{f \ \text{is continuous at every irrational and discontinuous at every rational of } (0,1)}
$$

**Key takeaway.** The set of continuity points is dense and so is its complement; finiteness of
$F$ is the entire engine, and it is what fails for Dirichlet's function in Problem L3.9.

In [34]:
from fractions import Fraction
from math import gcd

a_irr = np.sqrt(2) / 2                      # irrational point in (0,1)
eps = 0.05
N = int(np.ceil(1 / eps))                   # 1/N < eps needs N > 1/eps
F = [Fraction(p, q) for q in range(1, N + 1) for p in range(1, q) if gcd(p, q) == 1]
delta = min(abs(a_irr - float(r)) for r in F)
print(f"eps = {eps}   N = {N}   |F| = {len(F)} rationals with denominator <= N")
print(f"delta = min distance from a = {a_irr:.12f} to F  =  {delta:.12f}")

worst = 0.0
for q in range(1, 4000):
    for p in range(max(1, int((a_irr - delta) * q)), min(q, int((a_irr + delta) * q) + 2)):
        if gcd(p, q) == 1 and 0 < abs(a_irr - p / q) < delta:
            worst = max(worst, 1.0 / q)
print(f"largest f-value on 0 < |x - a| < delta among q < 4000 : {worst:.10f}   (< eps = {eps})")
assert worst < eps
assert all(abs(a_irr - float(r)) >= delta for r in F)

r0 = Fraction(1, 3)                          # a rational point: f jumps to 1/3 there
print(f"at the rational {r0}: f = {1/3:.6f}, but irrationals arbitrarily close give f = 0")
assert 1 / 3 > 0.0

eps = 0.05   N = 20   |F| = 127 rationals with denominator <= N
delta = min distance from a = 0.707106781187 to F  =  0.001224428245
largest f-value on 0 < |x - a| < delta among q < 4000 : 0.0243902439   (< eps = 0.05)
at the rational 1/3: f = 0.333333, but irrationals arbitrarily close give f = 0


### Problem L3.2 — Cesàro means inherit the limit

**Statement.** If $a_{n} \to L$, prove that
$c_{n} = \dfrac{a_{1} + \cdots + a_{n}}{n} \to L$.

**Intuition.** The finitely many early terms are diluted by $1/n$, and everything after them is
already within $\varepsilon/2$ of $L$.

**Solution.**

*Step 1 — split.* Given $\varepsilon \gt 0$ pick $N_{1}$ with
$\lvert a_{k} - L \rvert \lt \varepsilon/2$ for $k \gt N_{1}$, and write

$$
c_{n} - L = \frac{1}{n}\sum_{k=1}^{N_{1}} (a_{k} - L) + \frac{1}{n}\sum_{k=N_{1}+1}^{n} (a_{k} - L) .
$$

*Step 2 — the tail.* The second sum has at most $n - N_{1}$ terms, each below $\varepsilon/2$, so
its contribution is below $\varepsilon/2$.

*Step 3 — the head.* Put $M = \sum_{k=1}^{N_{1}} \lvert a_{k} - L \rvert$, a constant fixed by
$N_{1}$. Choose $N_{2}$ with $M/n \lt \varepsilon/2$ for $n \gt N_{2}$.

*Step 4 — combine.* For $n \gt \max(N_{1}, N_{2})$ the triangle inequality gives
$\lvert c_{n} - L \rvert \lt \varepsilon$.

$$
\boxed{a_{n} \to L \implies \frac{1}{n}\sum_{k=1}^{n} a_{k} \to L}
$$

**Key takeaway.** The converse is false — $a_{n} = (-1)^{n}$ has $c_{n} \to 0$ with no limit for
$a_{n}$ — so Cesàro summation is strictly stronger than ordinary convergence.

In [35]:
n_max = 200000
k = np.arange(1, n_max + 1)
a = 2.0 + (-1.0) ** k / np.sqrt(k)          # a_k -> 2
c = np.cumsum(a) / k
for n in (10, 100, 1000, 10000, 200000):
    print(f"  n = {n:7d}   a_n = {a[n-1]:.10f}   c_n = {c[n-1]:.10f}   |c_n - 2| = {abs(c[n-1]-2):.3e}")
assert abs(c[-1] - 2.0) < 1e-3

osc = (-1.0) ** k                            # the converse fails
print(f"  a_n = (-1)^n : c_n at n = 200000 is {np.cumsum(osc)[-1]/n_max:.6f}, "
      f"yet a_n has no limit")
assert abs(np.cumsum(osc)[-1] / n_max) < 1e-5

  n =      10   a_n = 2.3162277660   c_n = 1.9549274557   |c_n - 2| = 4.507e-02
  n =     100   a_n = 2.1000000000   c_n = 1.9944497636   |c_n - 2| = 5.550e-03
  n =    1000   a_n = 2.0316227766   c_n = 1.9994109088   |c_n - 2| = 5.891e-04
  n =   10000   a_n = 2.0100000000   c_n = 1.9999400101   |c_n - 2| = 5.999e-05
  n =  200000   a_n = 2.0022360680   c_n = 1.9999969811   |c_n - 2| = 3.019e-06
  a_n = (-1)^n : c_n at n = 200000 is 0.000000, yet a_n has no limit


### Problem L3.3 — Cauchy's functional equation with one point of continuity

**Statement.** Let $f : \mathbb{R} \to \mathbb{R}$ satisfy $f(x+y) = f(x) + f(y)$ for all $x, y$.
If $f$ is continuous at $0$, prove $f(x) = cx$ with $c = f(1)$.

**Intuition.** Additivity alone pins $f$ on the rationals; continuity at a single point transports
that to the reals by density.

**Solution.**

*Step 1 — the origin.* $f(0) = f(0+0) = 2f(0)$, so $f(0) = 0$, and
$0 = f(x + (-x)) = f(x) + f(-x)$ gives $f(-x) = -f(x)$.

*Step 2 — the integers.* Induction gives $f(nx) = nf(x)$ for $n \in \mathbb{N}$, and Step 1
extends it to $n \in \mathbb{Z}$.

*Step 3 — the rationals.* For $p \in \mathbb{Z}$ and $m \in \mathbb{N}$,
$p f(1) = f(p) = f\!\left( m \cdot \tfrac{p}{m} \right) = m f\!\left(\tfrac{p}{m}\right)$, so
$f(q) = cq$ for every $q \in \mathbb{Q}$ with $c = f(1)$.

*Step 4 — continuity everywhere from continuity at $0$.* For any $x$ and any $h$,
$f(x+h) - f(x) = f(h) \to f(0) = 0$ as $h \to 0$, so $f$ is continuous at *every* point.

*Step 5 — density.* Fix $x \in \mathbb{R}$ and take rationals $q_{n} \to x$. Theorem 4.4 applied
to the now-continuous $f$ gives $f(q_{n}) \to f(x)$, while $f(q_{n}) = c q_{n} \to cx$. Theorem 4.1
makes the two limits equal.

$$
\boxed{f(x) = cx \ \text{ for all } x \in \mathbb{R}, \quad c = f(1)}
$$

**Key takeaway.** Step 4 is the load-bearing one: additivity upgrades continuity at a *single*
point to continuity everywhere. Without any regularity the conclusion is false — a Hamel basis
produces additive functions whose graph is dense in $\mathbb{R}^{2}$.

In [36]:
c_const = 3.0


def f_add(x):
    return c_const * x


u = rng.uniform(-10, 10, 100000)
v = rng.uniform(-10, 10, 100000)
print("max |f(u+v) - f(u) - f(v)| :", np.abs(f_add(u + v) - f_add(u) - f_add(v)).max())
assert np.abs(f_add(u + v) - f_add(u) - f_add(v)).max() < 1e-9

target = np.sqrt(2)
p, q = 1, 1                                   # continued-fraction convergents of sqrt(2)
print("  rational q_n -> sqrt2                f(q_n) = 3 q_n     3 sqrt2")
for step in range(1, 15):
    p, q = p + 2 * q, p + q
    if step in (1, 2, 3, 6, 10, 14):
        qn = p / q
        print(f"  {p:10d}/{q:<10d} = {qn:.14f}   {f_add(qn):.12f}   {c_const*target:.12f}")
assert abs(f_add(p / q) - c_const * target) < 1e-9

max |f(u+v) - f(u) - f(v)| : 7.993605777301127e-15
  rational q_n -> sqrt2                f(q_n) = 3 q_n     3 sqrt2
           3/2          = 1.50000000000000   4.500000000000   4.242640687119
           7/5          = 1.40000000000000   4.200000000000   4.242640687119
          17/12         = 1.41666666666667   4.250000000000   4.242640687119
         239/169        = 1.41420118343195   4.242603550296   4.242640687119
        8119/5741       = 1.41421355164605   4.242640654938   4.242640687119
      275807/195025     = 1.41421356236380   4.242640687091   4.242640687119


### Problem L3.4 — An oscillatory sequence limit through an expansion

**Statement.** Compute $\lim_{n \to \infty} \cos\left( \pi \sqrt{n^{2}+n} \right)$ for
$n \in \mathbb{N}$.

**Intuition.** The square root is $n + \tfrac{1}{2}$ up to a vanishing correction, and
$\cos(\pi(n + \tfrac{1}{2}))$ is $0$; the correction controls how the value approaches it.

**Solution.**

*Step 1 — expand the root.*

$$
\sqrt{n^{2}+n} = n\left(1 + \frac{1}{n}\right)^{1/2} = n + \frac{1}{2} - \frac{1}{8n} + O(n^{-2}) .
$$

*Step 2 — multiply by $\pi$.*
$\pi\sqrt{n^{2}+n} = n\pi + \dfrac{\pi}{2} - \dfrac{\pi}{8n} + O(n^{-2})$.

*Step 3 — peel off the integer multiple.* $\cos(n\pi + \theta) = (-1)^{n}\cos\theta$, so the
sequence equals

$$
(-1)^{n} \cos\!\left( \frac{\pi}{2} - \frac{\pi}{8n} + O(n^{-2}) \right) = (-1)^{n} \sin\!\left( \frac{\pi}{8n} + O(n^{-2}) \right) .
$$

*Step 4 — squeeze.* The modulus is at most $\dfrac{\pi}{8n} + O(n^{-2}) \to 0$, and
$\lvert (-1)^{n} \rvert = 1$, so Theorem 4.3 applied to
$-\lvert s_{n} \rvert \le \cos(\pi\sqrt{n^{2}+n}) \le \lvert s_{n} \rvert$ gives the limit.

$$
\boxed{\lim_{n \to \infty} \cos\left( \pi\sqrt{n^{2}+n} \right) = 0, \qquad \text{with } \left\lvert \cdot \right\rvert \sim \frac{\pi}{8n}}
$$

**Key takeaway.** The alternating sign never enters the limit because it is bounded; only the
amplitude matters, and getting the amplitude requires the *second* term of the expansion, not the
first.

In [37]:
print("     n      cos(pi sqrt(n^2+n))     (-1)^n sin(pi/(8n))     ratio")
for n in (10, 100, 1000, 10000, 100000):
    exact = np.cos(np.pi * np.sqrt(n ** 2 + n))
    model = (-1.0) ** n * np.sin(np.pi / (8 * n))
    print(f"  {n:7d}   {exact:+.12e}   {model:+.12e}   {exact/model:.6f}")
    if n >= 1000:
        assert abs(exact / model - 1.0) < 1e-3
assert abs(np.cos(np.pi * np.sqrt(1e10 + 1e5))) < 1e-4

     n      cos(pi sqrt(n^2+n))     (-1)^n sin(pi/(8n))     ratio
       10   +3.741240526166e-02   +3.925981575907e-02   0.952944
      100   +3.907467785308e-03   +3.926980723806e-03   0.995031
     1000   +3.925028446164e-04   +3.926990716055e-04   0.999500
    10000   +3.926794610501e-05   +3.926990815978e-05   0.999950
   100000   +3.926958159854e-06   +3.926990816977e-06   0.999992


### Problem L3.5 — The decay rate of $a_{n+1} = \sin(a_{n})$

**Statement.** With $a_{1} = 1$ and $a_{n+1} = \sin(a_{n})$, prove $a_{n} \to 0$ and find
$\lim_{n \to \infty} \sqrt{n}\, a_{n}$.

**Intuition.** The sequence decreases to $0$, but only like $n^{-1/2}$, because each step removes
a *cubic* amount rather than a proportional one.

**Solution.**

*Step 1 — monotone and bounded.* For $t \in (0, \pi)$, $0 \lt \sin t \lt t$ (Proof 5.10), so
$0 \lt a_{n+1} \lt a_{n} \le 1$. A decreasing sequence bounded below converges, say to $L$.

*Step 2 — identify $L$.* $\sin$ is continuous, so passing to the limit in the recursion gives
$L = \sin L$, whose only solution in $[0,1]$ is $L = 0$.

*Step 3 — change variables.* Expand $\sin t = t - \tfrac{t^{3}}{6} + O(t^{5})$ and compute

$$
\frac{1}{\sin^{2} t} - \frac{1}{t^{2}} = \frac{1}{t^{2}}\left[ \left(1 - \frac{t^{2}}{6} + O(t^{4})\right)^{-2} - 1 \right] = \frac{1}{t^{2}}\left[ \frac{t^{2}}{3} + O(t^{4}) \right] = \frac{1}{3} + O(t^{2}) .
$$

*Step 4 — average.* Setting $t = a_{n} \to 0$ gives
$\dfrac{1}{a_{n+1}^{2}} - \dfrac{1}{a_{n}^{2}} \to \dfrac{1}{3}$. By Problem L3.2 applied to these
increments, their Cesàro mean has the same limit, and that mean telescopes:

$$
\frac{1}{n}\left( \frac{1}{a_{n+1}^{2}} - \frac{1}{a_{1}^{2}} \right) \longrightarrow \frac{1}{3} .
$$

*Step 5 — solve.* Hence $n a_{n}^{2} \to 3$ and $\sqrt{n}\,a_{n} \to \sqrt{3}$.

$$
\boxed{\lim_{n \to \infty} a_{n} = 0, \qquad \lim_{n \to \infty} \sqrt{n}\, a_{n} = \sqrt{3} \approx 1.7320508}
$$

**Key takeaway.** Convergence to $0$ says nothing about speed; the Stolz-Cesàro step converts a
statement about *increments of $1/a^{2}$* into the exact power law $a_{n} \sim \sqrt{3/n}$.

In [38]:
a_seq = 1.0
print("       n        a_n           sqrt(n) a_n")
for n in range(1, 200001):
    if n in (10, 100, 1000, 10000, 100000, 200000):
        print(f"  {n:8d}   {a_seq:.12f}   {np.sqrt(n)*a_seq:.10f}")
    a_seq = np.sin(a_seq)
print(f"  sqrt(3) = {np.sqrt(3):.10f}")
a_chk, n_chk = 1.0, 200000
for _ in range(n_chk - 1):
    a_chk = np.sin(a_chk)
assert abs(np.sqrt(n_chk) * a_chk - np.sqrt(3)) < 5e-3

       n        a_n           sqrt(n) a_n
        10   0.481329355262   1.5220970673
       100   0.169665324707   1.6966532471
      1000   0.054620126026   1.7272400433
     10000   0.017314486232   1.7314486232


    100000   0.005476997237   1.7319786006


    200000   0.003872898588   1.7320129023
  sqrt(3) = 1.7320508076


### Problem L3.6 — Every continuous self-map of $[0,1]$ has a fixed point

**Statement.** Let $f : [0,1] \to [0,1]$ be continuous. Prove there is $c \in [0,1]$ with
$f(c) = c$.

**Intuition.** The graph of $f$ starts on or above the diagonal and ends on or below it, so it
must cross.

**Solution.**

*Step 1 — the auxiliary function.* Put $g(x) = f(x) - x$, continuous on $[0,1]$ by Theorem 4.2.

*Step 2 — the endpoints.* $g(0) = f(0) \ge 0$ because $f$ takes values in $[0,1]$, and
$g(1) = f(1) - 1 \le 0$ for the same reason.

*Step 3 — the boundary cases.* If $g(0) = 0$ take $c = 0$; if $g(1) = 0$ take $c = 1$.

*Step 4 — the interior case.* Otherwise $g(1) \lt 0 \lt g(0)$, so $d = 0$ lies strictly between
$g(1)$ and $g(0)$ and Theorem 4.5 gives $c \in (0,1)$ with $g(c) = 0$.

$$
\boxed{\exists\, c \in [0,1] : f(c) = c \quad \text{(the one-dimensional Brouwer theorem)}}
$$

**Key takeaway.** Step 3 is not pedantry: Theorem 4.5 requires $d$ *strictly* between the endpoint
values, so the boundary cases have to be removed by hand before the theorem is invoked.

In [39]:
def f_self(x):
    return 0.5 * (1 + np.cos(np.pi * x))       # maps [0,1] onto [0,1]


def g_self(x):
    return f_self(x) - x


xs = np.linspace(0, 1, 100001)
print(f"range of f on [0,1] : [{f_self(xs).min():.6f}, {f_self(xs).max():.6f}]")
print(f"g(0) = {g_self(0.0):+.6f}   g(1) = {g_self(1.0):+.6f}")
c_fix = brentq(g_self, 0.0, 1.0, xtol=1e-15, rtol=8.9e-16)
print(f"fixed point c = {c_fix:.12f}   f(c) = {f_self(c_fix):.12f}   |f(c) - c| = {abs(f_self(c_fix)-c_fix):.3e}")
assert f_self(xs).min() >= -1e-15 and f_self(xs).max() <= 1 + 1e-15
assert abs(f_self(c_fix) - c_fix) < 1e-12

range of f on [0,1] : [0.000000, 1.000000]
g(0) = +1.000000   g(1) = -1.000000
fixed point c = 0.500000000000   f(c) = 0.500000000000   |f(c) - c| = 0.000e+00


### Problem L3.7 — $\sqrt{x}$ is uniformly continuous on $[0,\infty)$ and $x^{2}$ is not

**Statement.** Prove that $f(x) = \sqrt{x}$ is uniformly continuous on $[0,\infty)$ while
$g(x) = x^{2}$ is not.

**Intuition.** Uniform continuity forbids $\delta$ from shrinking as the base point moves; the
square root flattens out, the square steepens without bound.

**Solution.**

*Step 1 — the subadditivity inequality.* For $x \ge y \ge 0$,
$(\sqrt{x} - \sqrt{y})^{2} = x + y - 2\sqrt{xy} \le x - y$, because $y \le \sqrt{xy}$. Taking
square roots,

$$
\lvert \sqrt{x} - \sqrt{y} \rvert \le \sqrt{\lvert x - y \rvert} \qquad \text{for all } x, y \ge 0 .
$$

*Step 2 — a $\delta$ that ignores the base point.* Given $\varepsilon \gt 0$ take
$\delta = \varepsilon^{2}$. Then $\lvert x - y \rvert \lt \delta$ gives
$\lvert \sqrt{x} - \sqrt{y} \rvert \lt \varepsilon$, which is Definition 3.7.

*Step 3 — negate uniform continuity for $x^{2}$.* Take $\varepsilon = 1$. For any $\delta \gt 0$
set $x = 1/\delta$ and $y = 1/\delta + \delta/2$, so $\lvert x - y \rvert = \delta/2 \lt \delta$
while

$$
\lvert y^{2} - x^{2} \rvert = \lvert y - x \rvert (x+y) = \frac{\delta}{2}\left( \frac{2}{\delta} + \frac{\delta}{2} \right) = 1 + \frac{\delta^{2}}{4} \gt 1 .
$$

*Step 4 — no contradiction with Theorem 4.8.* $[0,\infty)$ is not compact, so Heine-Cantor does
not apply; on any $[0,R]$ it does, and Example 6.7 exhibits the resulting
$\delta(R) = \sqrt{R^{2}+1} - R \gt 0$.

$$
\boxed{\sqrt{\cdot} : \delta(\varepsilon) = \varepsilon^{2} \text{ on } [0,\infty); \qquad x^{2} : \text{no } \delta \text{ works, } \inf_{x \ge 0} \delta(x) = 0}
$$

**Key takeaway.** Uniform continuity is a property of the pair (function, domain). $\sqrt{\cdot}$
is uniformly continuous without being Lipschitz (Proposition 4.11), so the two notions genuinely
differ.

In [40]:
u = rng.uniform(0.0, 1e4, 200000)
v = rng.uniform(0.0, 1e4, 200000)
lhs = np.abs(np.sqrt(u) - np.sqrt(v))
rhs = np.sqrt(np.abs(u - v))
print(f"max (|sqrt u - sqrt v| - sqrt|u - v|) over 200000 pairs : {(lhs - rhs).max():.3e}  (<= 0)")
# The inequality is exact; the tolerance only absorbs rounding in sqrt,
# which is O(eps) relative to values of magnitude sqrt(1e4) = 100.
assert (lhs - rhs).max() <= 1e-10

# Squaring is not uniformly continuous on R: for every delta there are points
# closer than delta whose images differ by more than eps = 1. With x = 1/delta
# and y = x + delta/2 the gap is exactly 1 + delta^2/4, algebraically > 1.
eps1 = 1.0
print("\n  delta      x = 1/delta      |y^2 - x^2| computed      exact 1 + d^2/4")
for d in (1e-1, 1e-3, 1e-6):
    x = 1.0 / d
    y = x + d / 2
    computed = abs(y ** 2 - x ** 2)
    exact = 1.0 + d ** 2 / 4
    print(f"  {d:.0e}   {x:12.1f}   {computed:.12f}        {exact:.12f}")
    assert exact > eps1                 # the mathematics is strict
    assert computed >= eps1             # what double precision can still resolve

print("\n  Note: at delta = 1e-6 the excess is 2.5e-13 while one ulp at x^2 = 1e12")
print("  is about 1.2e-4, so the computed gap collapses to exactly 1.0. The")
print("  inequality is strict in R and only marginally observable in float64.")
print(f"\n  on [0, R] Heine-Cantor applies: delta(R) = sqrt(R^2+1) - R, e.g. R = 10 -> {np.sqrt(101)-10:.10f}")

max (|sqrt u - sqrt v| - sqrt|u - v|) over 200000 pairs : -9.091e-02  (<= 0)

  delta      x = 1/delta      |y^2 - x^2| computed      exact 1 + d^2/4
  1e-01           10.0   1.002500000000        1.002500000000
  1e-03         1000.0   1.000000249944        1.000000250000
  1e-06      1000000.0   1.000000000000        1.000000000000

  Note: at delta = 1e-6 the excess is 2.5e-13 while one ulp at x^2 = 1e12
  is about 1.2e-4, so the computed gap collapses to exactly 1.0. The
  inequality is strict in R and only marginally observable in float64.

  on [0, R] Heine-Cantor applies: delta(R) = sqrt(R^2+1) - R, e.g. R = 10 -> 0.0498756211


### Problem L3.8 — A nested radical

**Statement.** With $x_{1} = \sqrt{2}$ and $x_{n+1} = \sqrt{2 + x_{n}}$, compute
$\lim_{n \to \infty} x_{n}$.

**Intuition.** The map $t \mapsto \sqrt{2+t}$ pushes everything below $2$ upward and everything
above $2$ downward, so $2$ is the only possible destination.

**Solution.**

*Step 1 — bounded above.* By induction: $x_{1} = \sqrt2 \lt 2$, and $x_{k} \lt 2$ gives
$x_{k+1} = \sqrt{2+x_{k}} \lt \sqrt{4} = 2$.

*Step 2 — increasing.*

$$
x_{n+1}^{2} - x_{n}^{2} = 2 + x_{n} - x_{n}^{2} = -(x_{n}-2)(x_{n}+1) \gt 0
$$

because $0 \lt x_{n} \lt 2$. Both terms are positive, so $x_{n+1} \gt x_{n}$.

*Step 3 — converge.* An increasing sequence bounded above converges, by the completeness axiom
used as the least-upper-bound property (the same axiom that Proof 5.5 uses).

*Step 4 — identify the limit.* $\sqrt{2 + \cdot}$ is continuous, so
$L = \sqrt{2+L}$, that is $L^{2} - L - 2 = (L-2)(L+1) = 0$. Since $x_{n} \gt 0$, $L = 2$.

$$
\boxed{\lim_{n \to \infty} x_{n} = 2}
$$

**Key takeaway.** Passing to the limit inside $\sqrt{2 + \cdot}$ is Proposition 4.13: it needs
continuity of the outer map, which the square root has on $[0,\infty)$.

In [41]:
x_nest = np.sqrt(2.0)
print("   n         x_n                2 - x_n")
for n in range(1, 61):
    if n in (1, 2, 3, 5, 10, 20, 40, 60):
        print(f"  {n:3d}   {x_nest:.15f}   {2 - x_nest:.3e}")
    x_prev = x_nest
    x_nest = np.sqrt(2 + x_nest)
    assert x_nest > x_prev - 1e-15 and x_nest < 2 + 1e-15
print(f"  limit satisfies L = sqrt(2 + L): residual {abs(x_nest - np.sqrt(2 + x_nest)):.3e}")
assert abs(x_nest - 2.0) < 1e-12

   n         x_n                2 - x_n
    1   1.414213562373095   5.858e-01
    2   1.847759065022573   1.522e-01
    3   1.961570560806461   3.843e-02
    5   1.997590912410345   2.409e-03
   10   1.999997646903404   2.353e-06
   20   1.999999999997756   2.244e-12
   40   2.000000000000000   0.000e+00
   60   2.000000000000000   0.000e+00
  limit satisfies L = sqrt(2 + L): residual 0.000e+00


### Problem L3.9 — Dirichlet's function is nowhere continuous

**Statement.** For $D(x) = 1$ on $\mathbb{Q}$ and $D(x) = 0$ on $\mathbb{R}\setminus\mathbb{Q}$,
prove that $D$ is discontinuous at every real point.

**Intuition.** Both $\mathbb{Q}$ and its complement are dense, so every neighbourhood contains
points where $D$ takes both values.

**Solution.**

*Step 1 — fix $\varepsilon$.* Take $\varepsilon = \tfrac{1}{2}$ and let $a \in \mathbb{R}$ be
arbitrary.

*Step 2 — rational centre.* If $a \in \mathbb{Q}$ then $D(a) = 1$. Every interval
$(a-\delta, a+\delta)$ contains an irrational $x$ — for instance $a + \delta/(2\sqrt{2})$, which is
irrational because $\delta/(2\sqrt2)$ is — and $\lvert D(x) - D(a) \rvert = 1 \ge \varepsilon$.

*Step 3 — irrational centre.* If $a \notin \mathbb{Q}$ then $D(a) = 0$, and every interval
contains a rational $x$ — for instance $\lfloor a/\delta' \rfloor \delta'$ with
$\delta' = \delta/2$ rational — giving $\lvert D(x) - D(a) \rvert = 1 \ge \varepsilon$.

*Step 4 — conclude.* No $\delta$ works at any $a$, so Definition 3.4 fails everywhere.

$$
\boxed{D \ \text{is discontinuous at every } a \in \mathbb{R}; \ \limsup_{x \to a} D = 1, \ \liminf_{x \to a} D = 0}
$$

**Key takeaway.** Contrast Problem L3.1: Thomae's function works because only *finitely many*
rationals carry a large value, whereas $D$ gives all of them the value $1$, and no finite set can
be dodged.

In [42]:
import sympy as sp


def dirichlet(x):
    """Exact classification via sympy, not a floating-point test."""
    return 1 if sp.nsimplify(x).is_rational else 0


for a, name in [(sp.Rational(1, 2), "a = 1/2 (rational)"), (sp.sqrt(2) / 2, "a = sqrt(2)/2 (irrational)")]:
    print(f"{name}:  D(a) = {dirichlet(a)}")
    for delta in (sp.Rational(1, 10), sp.Rational(1, 1000)):
        witness = a + delta / (2 * sp.sqrt(2)) if dirichlet(a) == 1 else sp.Rational(sp.floor(a / (delta / 2))) * delta / 2
        print(f"   delta = {float(delta):.4f}  witness x = {float(witness):.10f}"
              f"   D(x) = {dirichlet(witness)}   |D(x) - D(a)| = {abs(dirichlet(witness)-dirichlet(a))}")
        assert abs(dirichlet(witness) - dirichlet(a)) == 1
        assert abs(float(witness - a)) < float(delta)

a = 1/2 (rational):  D(a) = 1
   delta = 0.1000  witness x = 0.5353553391   D(x) = 0   |D(x) - D(a)| = 1
   delta = 0.0010  witness x = 0.5003535534   D(x) = 0   |D(x) - D(a)| = 1
a = sqrt(2)/2 (irrational):  D(a) = 0
   delta = 0.1000  witness x = 0.7000000000   D(x) = 1   |D(x) - D(a)| = 1
   delta = 0.0010  witness x = 0.7070000000   D(x) = 1   |D(x) - D(a)| = 1


### Problem L3.10 — The $n$-th root of a factorial

**Statement.** Compute $\displaystyle \lim_{n \to \infty} \frac{(n!)^{1/n}}{n}$ using Stirling's
formula $n! \sim \sqrt{2\pi n}\,(n/e)^{n}$.

**Intuition.** Taking an $n$-th root kills every sub-exponential factor, leaving $n/e$.

**Solution.**

*Step 1 — substitute Stirling.*

$$
(n!)^{1/n} \sim \left( \sqrt{2\pi n} \left(\frac{n}{e}\right)^{n} \right)^{1/n} = (2\pi n)^{1/(2n)} \cdot \frac{n}{e} .
$$

*Step 2 — divide by $n$.* $\dfrac{(n!)^{1/n}}{n} \sim \dfrac{(2\pi n)^{1/(2n)}}{e}$.

*Step 3 — kill the prefactor.* Its logarithm is $\dfrac{\ln(2\pi n)}{2n} \to 0$, and $\exp$ is
continuous at $0$, so by Proposition 4.13 the prefactor tends to $1$.

*Step 4 — conclude.*

$$
\boxed{\lim_{n \to \infty} \frac{(n!)^{1/n}}{n} = \frac{1}{e} \approx 0.3678794412}
$$

**Key takeaway.** The convergence is slow — the correction is $\Theta(\ln n / n)$ — which is why a
numerical check must go to $n$ in the thousands before the third digit settles.

In [43]:
print("      n     (n!)^(1/n)/n      1/e        error")
for n in (10, 100, 1000, 10000, 100000):
    val = np.exp(lgamma(n + 1) / n) / n
    print(f"  {n:7d}   {val:.12f}   {1/np.e:.12f}   {abs(val - 1/np.e):.3e}")
val_big = np.exp(lgamma(100001) / 100000) / 100000
print(f"  model 1/e * (2 pi n)^(1/(2n)) at n = 1e5 : "
      f"{np.exp(np.log(2*np.pi*1e5)/2e5)/np.e:.12f}")
assert abs(val_big - 1 / np.e) < 1e-4

      n     (n!)^(1/n)/n      1/e        error
       10   0.452872868812   0.367879441171   8.499e-02
      100   0.379926893448   0.367879441171   1.205e-02
     1000   0.369491663472   0.367879441171   1.612e-03
    10000   0.368082718222   0.367879441171   2.033e-04
   100000   0.367903999423   0.367879441171   2.456e-05
  model 1/e * (2 pi n)^(1/(2n)) at n = 1e5 : 0.367903999420


### Problem L3.11 — A finite limit at infinity forces uniform continuity

**Statement.** Let $f$ be continuous on $[0,\infty)$ with $\lim_{x \to \infty} f(x) = L$ finite.
Prove $f$ is uniformly continuous on $[0,\infty)$.

**Intuition.** Far out, $f$ is nearly constant, so any two nearby points have nearly equal values;
near the origin the domain is compact and Theorem 4.8 applies.

**Solution.**

*Step 1 — the tail.* Given $\varepsilon \gt 0$, Definition 3.3 supplies $X \gt 0$ with
$\lvert f(x) - L \rvert \lt \varepsilon/2$ for all $x \ge X$. Hence for $x, y \ge X$,

$$
\lvert f(x) - f(y) \rvert \le \lvert f(x) - L \rvert + \lvert L - f(y) \rvert \lt \varepsilon .
$$

*Step 2 — the compact head.* $f$ is continuous on the closed bounded interval $[0, X+1]$, so
Theorem 4.8 gives $\delta_{1} \gt 0$ with
$\lvert f(x) - f(y) \rvert \lt \varepsilon$ for $x, y \in [0, X+1]$ and
$\lvert x - y \rvert \lt \delta_{1}$.

*Step 3 — one $\delta$ for both.* Put $\delta = \min(\delta_{1}, 1)$ and take any
$x, y \ge 0$ with $\lvert x - y \rvert \lt \delta$. If both lie in $[0, X+1]$, Step 2 applies. If
not, one of them exceeds $X+1$, and since they differ by less than $1$ both exceed $X$, so Step 1
applies.

$$
\boxed{f \ \text{continuous on } [0,\infty) \ \text{with a finite limit at } \infty \implies f \ \text{uniformly continuous on } [0,\infty)}
$$

**Key takeaway.** Compactness is not necessary for uniform continuity, only sufficient: what the
proof really needs is that the function stops varying outside a compact set. This is exactly why
$\sigma$, $\tanh$ and $\operatorname{arctan}$ are uniformly continuous on all of $\mathbb{R}$ while
$x^{2}$ is not.

In [44]:
def f_tail(x):
    return np.arctan(x)                        # continuous, limit pi/2 at infinity


eps_uc = 1e-3
X = np.tan(np.pi / 2 - eps_uc / 2)             # |f(x) - L| < eps/2 for x >= X
print(f"eps = {eps_uc}   X = {X:.4f}   |arctan(X) - pi/2| = {abs(np.arctan(X) - np.pi/2):.3e}")

delta = eps_uc                                  # arctan is 1-Lipschitz, so delta = eps works
xs = np.linspace(0.0, 5 * X, 400001)
gap = np.abs(f_tail(xs + delta) - f_tail(xs)).max()
print(f"delta = {delta:.1e}   sup |f(x+delta) - f(x)| over [0, 5X] = {gap:.6e}   (< eps)")
assert gap < eps_uc

bad = np.abs((xs + delta) ** 2 - xs ** 2).max()
print(f"for x^2 the same delta gives sup |f(x+delta) - f(x)| = {bad:.3f} on the same range: "
      f"no finite limit at infinity, no uniform continuity")
assert bad > 1.0

eps = 0.001   X = 1999.9998   |arctan(X) - pi/2| = 5.000e-04
delta = 1.0e-03   sup |f(x+delta) - f(x)| over [0, 5X] = 9.999997e-04   (< eps)
for x^2 the same delta gives sup |f(x+delta) - f(x)| = 20.000 on the same range: no finite limit at infinity, no uniform continuity


### Problem L3.12 — The continuous image of a compact interval is a compact interval

**Statement.** Let $f : [a,b] \to \mathbb{R}$ be continuous with $a \lt b$. Prove
$f([a,b]) = [m, M]$ where $m = \min f$ and $M = \max f$.

**Intuition.** The Extreme Value Theorem produces the endpoints of the image; the Intermediate
Value Theorem fills in everything between them.

**Solution.**

*Step 1 — the endpoints exist.* Theorem 4.7 gives $x_{\min}, x_{\max} \in [a,b]$ with
$f(x_{\min}) = m$ and $f(x_{\max}) = M$, both attained.

*Step 2 — the image is contained in $[m, M]$.* By definition of minimum and maximum,
$m \le f(x) \le M$ for every $x$, so $f([a,b]) \subseteq [m,M]$.

*Step 3 — the image contains $[m,M]$.* If $m = M$ the function is constant and there is nothing
to prove. Otherwise let $d \in (m, M)$ and let $I$ be the closed interval with endpoints
$x_{\min}$ and $x_{\max}$; then $I \subseteq [a,b]$, $f$ is continuous on $I$, and $d$ lies
strictly between $f(x_{\min}) = m$ and $f(x_{\max}) = M$. Theorem 4.5 applied on $I$ produces
$c$ in the interior of $I$ with $f(c) = d$.

*Step 4 — combine.* Both inclusions hold, and $m$, $M$ are themselves attained by Step 1.

$$
\boxed{f([a,b]) = \left[ \min_{[a,b]} f, \ \max_{[a,b]} f \right]}
$$

**Key takeaway.** This is the single statement behind both hard theorems of the module: continuity
preserves compactness (Theorem 4.7) and connectedness (Theorem 4.5), and an interval is exactly a
set that is both.

In [45]:
def f_img(x):
    return np.sin(3 * x) + 0.5 * x


grid = np.linspace(0.0, 2.0, 400001)
m_val, M_val = f_img(grid).min(), f_img(grid).max()
x_min, x_max = grid[int(np.argmin(f_img(grid)))], grid[int(np.argmax(f_img(grid)))]
print(f"min f = {m_val:.10f} at x = {x_min:.6f}     max f = {M_val:.10f} at x = {x_max:.6f}")

lo_x, hi_x = sorted((x_min, x_max))
for d in np.linspace(m_val + 1e-6, M_val - 1e-6, 5):
    c = brentq(lambda t: f_img(t) - d, lo_x, hi_x, xtol=1e-14)
    print(f"  target d = {d:+.8f}   solved f(c) = {f_img(c):+.8f} at c = {c:.8f}")
    assert abs(f_img(c) - d) < 1e-10
assert m_val <= f_img(grid).min() and f_img(grid).max() <= M_val

min f = -0.2285231470 at x = 1.514980     max f = 1.2757206982 at x = 0.579415
  target d = -0.22852215   solved f(c) = -0.22852215 at c = 1.51450554
  target d = +0.14753831   solved f(c) = +0.14753831 at c = 1.20455787
  target d = +0.52359878   solved f(c) = +0.52359878 at c = 1.04719755
  target d = +0.89965924   solved f(c) = +0.89965924 at c = 0.88983723
  target d = +1.27571970   solved f(c) = +1.27571970 at c = 0.57988956
